# Test checkpoint GSD train trên B-Free `samecat_origBG` — **CHỈ Tiny-GenImage 9 case**

Dùng **đúng evaluator Tiny đã chấm GSD-aug-Tiny** (runtime byte-identical với
`test_gsd_uploaded_best_variants_kaggle.ipynb`), chỉ thêm cell nhận checkpoint B-Free.
Cùng 7.000 ảnh test, cùng thứ tự, batch 8, threshold 0.5, cùng 9 case → so trực tiếp được.
Cuối notebook tự in bảng so sánh với GSD-aug-Tiny. **Không chấm B-Free.**

## Input cần Add (3 dataset) + GPU T4 + Internet
1. `hoangha071005/check-point-gsd-train-bfee-samecat-ori` (mặc định đọc `best.pt` = VE2).
2. `yangsangtai/tiny-genimage`.
3. `hadinho0710/mainfest-train-genimage` (cần `seed42/test_manifest.csv`).
KHÔNG cần B-Free. Rồi **Save Version → Save & Run All**.

## Case (biến đổi trên RGB gốc, trước crop/pad)
`native`, `fake_jpeg96` (chỉ fake nén), `all_jpeg96/90/80`, `resize_square256/560`, `blur_sigma1`, `resize256_jpeg96`.

## Thời gian / resume
Ước lượng ~3–4 giờ trên T4 (chưa đo trực tiếp). Nếu hết giờ: Add Input Output phiên này, Save & Run All lại
→ tiếp tục từ batch đã lưu (cùng GPU/thư viện). Chấm `checkpoint_ve1.pt`: đổi `CHECKPOINT_NAME`, output tách riêng.
Với checkpoint này Tiny là **cross-dataset** (GSD-aug-Tiny được train trên Tiny).

In [ ]:
%pip install -q "transformers==4.46.3" "huggingface_hub>=0.23.2,<1.0" "pandas<3" "scikit-learn<1.9" "pillow<12" matplotlib tqdm

## Runtime evaluator Tiny (byte-identical với notebook đã chấm GSD-aug-Tiny)

In [ ]:
"""Paired full-CLIP/GSD experiment. Self-contained Kaggle runtime.

Paper v2 Fig.3: GSD is BEFORE each of the last four transformer blocks.
No fake-only codec transformations, test tuning, or automatic batch-size changes.
"""
from __future__ import annotations

import copy
import gc
import hashlib
import io
import json
import math
import platform
import random
from pathlib import Path

import numpy as np
import pandas as pd
import PIL
from PIL import Image, ImageFilter, ImageOps
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.checkpoint import checkpoint
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, balanced_accuracy_score, confusion_matrix, average_precision_score
from transformers import CLIPVisionModel, CLIPVisionConfig
from tqdm.auto import tqdm


DEFAULT_CONFIG = dict(
    dataset_root=None, input_root='/kaggle/input', output_root='/kaggle/working/clip_gsd_tiny',
    model_id='openai/clip-vit-large-patch14', local_files_only=False,
    seeds=[42], models=['clip_gsd'], epochs=50,
    early_stopping=True, patience=3, min_delta=0.001,
    train_batch=128, test_batch=512, micro_batch=2, anchor_micro_batch=8,
    lr=1e-6, head_lr=1e-6, modulation_lr=1e-6, weight_decay=0.01,
    betas=[0.9, 0.999], eps=1e-8, k=64, lambda_max=1., a_init=2., b_init=1.,
    validation_fraction=0.1, num_workers=2, gradient_checkpointing=True,
    precision='fp16', svd_driver='gesvdj', resume=True,
    jpeg_probability=0.5, jpeg_train_quality=[75, 100], jpeg_subsampling=0,
    blur_probability=0.5, blur_sigma=[0., 2.], horizontal_flip_probability=0.5,
    jpeg_eval_qualities=[96], size_control=256, threshold=0.5,
    pooling='patch_mean_post_ln', source_generator='sd15', strict_eight_generators=False,
    expected_generators=['sd15', 'adm', 'biggan', 'glide', 'midjourney', 'wukong', 'vqdm'],
    smoke=False, smoke_train_per_class=64, smoke_eval_per_class=4,
)


def dump(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, indent=2, ensure_ascii=False, default=str), encoding='utf-8')


def digest(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, default=str).encode()).hexdigest()


def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def canonical_generator(name):
    n = ''.join(c for c in name.lower() if c.isalnum())
    aliases = {'sd14': ('sd14', 'sdv14', 'sdv4', 'stablediffusionv14', 'stablediffusion14'),
               'sd15': ('sd15', 'sdv15', 'sdv5', 'stablediffusionv15', 'stablediffusion15'),
               'adm': ('adm', 'guided'), 'biggan': ('biggan',), 'glide': ('glide',),
               'midjourney': ('midjourney',), 'wukong': ('wukong',), 'vqdm': ('vqdm',)}
    matches = [key for key, values in aliases.items() if any(v in n for v in values)]
    return matches[0] if len(matches) == 1 else None


def find_root(cfg):
    if cfg['dataset_root']:
        return Path(cfg['dataset_root'])
    roots = set()
    for p in Path(cfg['input_root']).rglob('*'):
        if p.is_dir() and canonical_generator(p.name):
            if any((p / s).is_dir() for s in ('train', 'training')):
                roots.add(p.parent)
    if len(roots) != 1:
        raise ValueError(f'Set dataset_root explicitly. Candidate roots: {sorted(map(str, roots))}')
    return roots.pop()


def scan_dataset(cfg, out):
    root = find_root(cfg)
    rows, errors = [], []
    extensions = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}
    for g in sorted(root.iterdir()):
        gen = canonical_generator(g.name)
        if not g.is_dir() or gen is None:
            continue
        for split, names in [('train', ['train', 'training']), ('test', ['val', 'valid', 'validation', 'test'])]:
            available = [g / s for s in names if (g / s).is_dir()]
            if len(available) != 1:
                raise ValueError(f'{g}: expected exactly one {split} split, found {available}')
            labels = {0: [], 1: []}
            mapping = {'nature': 0, 'real': 0, '0_real': 0, 'ai': 1, 'fake': 1, '1_fake': 1}
            for folder in available[0].iterdir():
                if folder.is_dir() and folder.name.lower() in mapping:
                    labels[mapping[folder.name.lower()]].append(folder)
            if any(len(v) != 1 for v in labels.values()):
                raise ValueError(f'Unexpected label folders: {available[0]} {labels}')
            for label, folders in labels.items():
                for path in sorted(folders[0].rglob('*')):
                    if path.suffix.lower() not in extensions or not path.is_file():
                        continue
                    # Audit source-train only: other generators' train images are not used.
                    if split == 'train' and gen not in cfg.get('train_generators', [cfg['source_generator']]):
                        continue
                    try:
                        raw = path.read_bytes()
                        with Image.open(io.BytesIO(raw)) as im:
                            fmt, w, h = im.format, *im.size
                            qtable = im.quantization if hasattr(im, 'quantization') else None
                            rgb = im.convert('RGB')
                            pixel_hash = hashlib.sha256(f'{w}x{h}:'.encode() + rgb.tobytes()).hexdigest()
                        rel = path.relative_to(root).as_posix()
                        rows.append(dict(sample_id=hashlib.sha256(rel.encode()).hexdigest(),
                                         path=str(path), relative_path=rel, generator=gen, split=split,
                                         label=label, width=w, height=h, format=fmt,
                                         pixel_hash=pixel_hash, file_hash=hashlib.sha256(raw).hexdigest(),
                                         qtable_hash=digest(qtable) if qtable else '', file_bytes=len(raw)))
                        if cfg.get('scan_log_every') and len(rows) % cfg['scan_log_every'] == 0:
                            print(f'[scan] {len(rows):,} images; {gen}/{split}; invalid={len(errors)}', flush=True)
                    except Exception as e:
                        errors.append(dict(path=str(path), error=repr(e)))
    df = pd.DataFrame(rows)
    pd.DataFrame(errors).to_csv(out / 'invalid_images.csv', index=False)
    if df.empty:
        raise ValueError('No usable images')
    if (df.groupby('pixel_hash').label.nunique() > 1).any():
        raise ValueError('Identical decoded pixels have conflicting labels; inspect dataset')
    df.to_csv(out / 'image_audit.csv', index=False)
    df.groupby(['split', 'generator', 'label', 'format']).size().rename('count').to_csv(out / 'format_counts.csv')
    found = set(df.loc[df.split == 'test', 'generator'])
    expected = set(cfg.get('expected_generators') or [])
    if not cfg['smoke'] and expected and found != expected:
        raise ValueError(f'Test generators mismatch: missing={sorted(expected-found)}, '
                         f'unexpected={sorted(found-expected)}, found={sorted(found)}. '
                         'Inspect image_audit.csv and invalid_images.csv.')
    if cfg['strict_eight_generators'] and len(found) != 8:
        raise ValueError(f'Expected eight test generators; found {sorted(found)}')
    for source_gen in cfg.get('train_generators', [cfg['source_generator']]):
        source = df[(df.split == 'train') & (df.generator == source_gen)]
        if set(source.label) != {0, 1}:
            raise ValueError(f"Source {source_gen!r} needs real and fake training images. "
                             'yangsangtai/tiny-genimage excludes SD14; use source_generator=sd15.')
    return df


def balanced(frame, seed, limit=None):
    n = int(frame.label.value_counts().reindex([0, 1], fill_value=0).min())
    n = min(n, limit) if limit else n
    if n == 0:
        raise ValueError('Every cohort must contain real and fake')
    return pd.concat([frame[frame.label == y].sample(n=n, random_state=seed + y) for y in (0, 1)])


def make_splits(df, cfg, seed, out):
    source = df[(df.split == 'train') & (df.generator == cfg['source_generator'])].drop_duplicates('pixel_hash')
    if cfg['smoke']:
        source = balanced(source, seed, cfg['smoke_train_per_class'])
    source = balanced(source, seed)
    tr, va = train_test_split(source, test_size=cfg['validation_fraction'], stratify=source.label, random_state=seed)
    used = set(source.pixel_hash)
    tests = df[df.split == 'test'].copy()
    removed = tests[tests.pixel_hash.isin(used)]
    removed.to_csv(out / 'excluded_test_overlap.csv', index=False)
    tests = tests[~tests.pixel_hash.isin(used)]
    tests = tests.drop_duplicates(['generator', 'pixel_hash'])
    tests = pd.concat([balanced(g, seed, cfg['smoke_eval_per_class'] if cfg['smoke'] else None)
                       for _, g in tests.groupby('generator')])
    # Label-blind order shared by all models and processing cases; never sort batches by label.
    def ordered(frame):
        frame = frame.copy()
        frame['_order'] = frame.sample_id.map(lambda s: digest([seed, s]))
        return frame.sort_values('_order').drop(columns='_order').reset_index(drop=True)
    tr, va, tests = ordered(tr), ordered(va), ordered(tests)
    for name, frame in [('train', tr), ('validation', va), ('test', tests)]:
        frame.to_csv(out / f'{name}_manifest.csv', index=False)
    dump(out / 'split_summary.json', dict(train=len(tr), validation=len(va), test=len(tests),
         test_overlap_removed=len(removed), test_repeated_real_pixels=int(tests[tests.label == 0].pixel_hash.duplicated().sum()),
         train_hash=digest(tr.sample_id.tolist()), validation_hash=digest(va.sample_id.tolist()),
         test_hash=digest(tests.sample_id.tolist()), source_generator=cfg['source_generator'],
         selection=f"source {cfg['source_generator']} train only; no test selection"))
    return tr, va, tests


def jpeg(image, quality, subsampling):
    buf = io.BytesIO()
    image.save(buf, format='JPEG', quality=int(quality), subsampling=subsampling,
               optimize=False, progressive=False)
    buf.seek(0)
    with Image.open(buf) as decoded:
        return decoded.convert('RGB')


def processing_cases(cfg):
    return ['native', 'size'] + [c for q in cfg['jpeg_eval_qualities'] for c in (f'jpeg{q}', f'size_jpeg{q}')]


def process(image, case, cfg):
    # Size stress-test includes square warp; explicitly NOT claimed to remove all size/history bias.
    if case.startswith('size'):
        image = image.resize((cfg['size_control'], cfg['size_control']), Image.Resampling.BICUBIC)
    if 'jpeg' in case:
        quality = int(case.split('jpeg')[-1])
        image = jpeg(image, quality, cfg['jpeg_subsampling'])
    return image


class Images(Dataset):
    def __init__(self, frame, cfg, seed, train=False, epoch=0, case='native'):
        self.rows = frame.to_dict('records')
        self.cfg, self.seed, self.train, self.epoch, self.case = cfg, seed, train, epoch, case

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, i):
        r, c = self.rows[i], self.cfg
        with Image.open(r['path']) as im:
            image = im.convert('RGB')
        if self.train:
            rng = random.Random(int(digest([self.seed, self.epoch, r['sample_id']])[:16], 16))
            if rng.random() < c['horizontal_flip_probability']:
                image = ImageOps.mirror(image)
            if rng.random() < c['blur_probability']:
                image = image.filter(ImageFilter.GaussianBlur(rng.uniform(*c['blur_sigma'])))
            if rng.random() < c['jpeg_probability']:
                image = jpeg(image, rng.randint(*c['jpeg_train_quality']), c['jpeg_subsampling'])
        else:
            image = process(image, self.case, c)
        # Explicit standard OpenAI CLIP: shorter-side 224 bicubic, center crop, RGB mean/std.
        w, h = image.size
        image = image.resize((224 if w <= h else int(224 * w / h),
                              224 if h <= w else int(224 * h / w)), Image.Resampling.BICUBIC)
        w, h = image.size
        left, top = int(round((w - 224) / 2)), int(round((h - 224) / 2))
        image = image.crop((left, top, left + 224, top + 224))
        x = torch.from_numpy(np.asarray(image, dtype=np.float32).copy()).permute(2, 0, 1) / 255.
        x = (x - torch.tensor([.48145466, .4578275, .40821073])[:, None, None]) / torch.tensor([.26862954, .26130258, .27577711])[:, None, None]
        return x, int(r['label']), i


def loader(frame, cfg, seed, train=False, epoch=0, case='native'):
    ds = Images(frame, cfg, seed, train, epoch, case)
    return DataLoader(ds, batch_size=cfg['train_batch'] if train else cfg['test_batch'],
                      shuffle=train, generator=torch.Generator().manual_seed(seed + epoch),
                      num_workers=cfg['num_workers'], pin_memory=torch.cuda.is_available(),
                      drop_last=train)


class SemanticSuppression(nn.Module):
    def __init__(self, a=2., b=1., maximum=1.):
        super().__init__()
        self.a, self.b = nn.Parameter(torch.tensor(float(a))), nn.Parameter(torch.tensor(float(b)))
        self.maximum = maximum

    def forward(self, features, basis):
        # FP32 projection/occupancy; do not detach occupancy (Eq.10 differentiated end-to-end).
        with torch.autocast(features.device.type, enabled=False):
            f = features.float()
            p = (f @ basis.float()) @ basis.float().T
            r = p.square().sum((1, 2)) / f.square().sum((1, 2)).clamp_min(1e-12)
            lam = self.maximum * torch.sigmoid(self.a * r + self.b)
            return (f - lam[:, None, None] * p).to(features.dtype)


def top_basis(matrix, k, driver=None):
    with torch.autocast(matrix.device.type, enabled=False):
        options = {'driver': driver} if matrix.is_cuda and driver else {}
        _, _, vh = torch.linalg.svd(matrix.float(), full_matrices=False, **options)
        if k > min(matrix.shape):
            raise ValueError('k exceeds available dimensions')
        return vh[:k].T.contiguous().detach()


class Detector(nn.Module):
    def __init__(self, vision, cfg, gsd):
        super().__init__()
        self.vision = vision
        self.cfg = cfg
        self.selected = tuple(range(len(vision.encoder.layers) - 4, len(vision.encoder.layers)))
        d = vision.config.hidden_size
        self.head = nn.Linear(d, 2)
        self.gsd = nn.ModuleDict({str(l): SemanticSuppression(cfg['a_init'], cfg['b_init'], cfg['lambda_max'])
                                  for l in self.selected}) if gsd else nn.ModuleDict()

    def forward(self, x, bases=None):
        h = self.vision.pre_layrnorm(self.vision.embeddings(x))
        for l, block in enumerate(self.vision.encoder.layers):
            if str(l) in self.gsd:
                h = self.gsd[str(l)](h, bases[l])
            def apply(t, layer=block):
                return layer(t, attention_mask=None, causal_attention_mask=None)[0]
            h = checkpoint(apply, h, use_reentrant=False) if self.training and self.cfg['gradient_checkpointing'] else apply(h)
        h = self.vision.post_layernorm(h)
        if self.cfg['pooling'] == 'patch_mean_post_ln':
            h = h[:, 1:].mean(1)
        elif self.cfg['pooling'] == 'all_mean_post_ln':
            h = h.mean(1)
        else:
            raise ValueError('Unsupported pooling')
        return self.head(h)


def amp(cfg, device):
    return torch.autocast(device.type, dtype=torch.float16 if cfg['precision'] == 'fp16' else torch.bfloat16,
                          enabled=device.type == 'cuda' and cfg['precision'] != 'fp32')


@torch.no_grad()
def estimate_bases(anchor, images, cfg, device, selected):
    # Anchor PRE-block states, matching Fig.3. Collect the ENTIRE logical batch on CPU.
    chunks = {l: [] for l in selected}
    for x in images.split(cfg['anchor_micro_batch']):
        with amp(cfg, device):
            h = anchor.pre_layrnorm(anchor.embeddings(x.to(device)))
            for l, layer in enumerate(anchor.encoder.layers):
                if l in chunks:
                    chunks[l].append(h[:, 1:].detach().cpu())
                h = layer(h, attention_mask=None, causal_attention_mask=None)[0]
    bases = {}
    for l in selected:
        m = torch.cat(chunks.pop(l), dim=0).flatten(0, 1).to(device=device, dtype=torch.float32)
        bases[l] = top_basis(m, cfg['k'], cfg['svd_driver'])
        del m
    return bases


def build_models(cfg, device, gsd):
    pretrained = CLIPVisionModel.from_pretrained(cfg['model_id'], local_files_only=cfg['local_files_only'], attn_implementation='eager')
    v = pretrained.vision_model
    assert (v.config.hidden_size, v.config.num_hidden_layers, v.config.patch_size, v.config.image_size) == (1024, 24, 14, 224), 'Expected CLIP ViT-L/14 224'
    model = Detector(v, cfg, gsd).to(device)
    anchor = copy.deepcopy(v).requires_grad_(False).eval().to(device) if gsd else None
    if anchor is not None and device.type == 'cuda' and cfg['precision'] != 'fp32':
        anchor.to(dtype=torch.float16 if cfg['precision'] == 'fp16' else torch.bfloat16)
    return model, anchor


@torch.no_grad()
def predict(model, anchor, frame, cfg, seed, device, case):
    model.eval()
    result = np.full(len(frame), np.nan)
    batches = np.full(len(frame), -1, dtype=np.int64)
    for batch_id, (images, _, indices) in enumerate(tqdm(loader(frame, cfg, seed, case=case), desc=case)):
        bases = estimate_bases(anchor, images, cfg, device, model.selected) if anchor is not None else None
        for start in range(0, len(images), cfg['micro_batch']):
            ix = indices[start:start + cfg['micro_batch']].numpy()
            with amp(cfg, device):
                logits = model(images[start:start + cfg['micro_batch']].to(device), bases)
            result[ix] = logits.float().softmax(-1)[:, 1].cpu().numpy()
            batches[ix] = batch_id
    if not np.isfinite(result).all():
        raise ValueError('Nonfinite or missing predictions')
    return result, batches


def metric(y, p, threshold):
    pred = p >= threshold
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return dict(n=len(y), auc=float(roc_auc_score(y, p)), ap=float(average_precision_score(y, p)),
                balanced_accuracy=float(balanced_accuracy_score(y, pred)), accuracy=float(np.mean(y == pred)),
                fake_recall=float(tp / (tp + fn)), real_recall=float(tn / (tn + fp)),
                tn=int(tn), fp=int(fp), fn=int(fn), tp=int(tp))


class EarlyStopping:
    """Track significant validation-loss improvements, separately from best checkpoint."""
    def __init__(self, enabled=True, patience=3, min_delta=0.001, state=None):
        if patience < 1 or min_delta < 0:
            raise ValueError('patience must be >=1; min_delta must be >=0')
        self.enabled, self.patience, self.min_delta = enabled, patience, min_delta
        self.state = dict(reference_loss=float('inf'), bad_epochs=0, stopped=False)
        if state is not None:
            self.state.update(state)

    def update(self, loss):
        if not math.isfinite(loss):
            raise FloatingPointError('Nonfinite validation loss')
        if loss < self.state['reference_loss'] - self.min_delta:
            self.state.update(reference_loss=loss, bad_epochs=0)
        else:
            self.state['bad_epochs'] += 1
        self.state['stopped'] = bool(self.enabled and self.state['bad_epochs'] >= self.patience)
        return self.state['stopped']


def train_model(model, anchor, train, val, cfg, seed, device, folder, signature):
    groups = [dict(params=model.vision.parameters(), lr=cfg['lr']),
              dict(params=model.head.parameters(), lr=cfg['head_lr'])]
    if len(model.gsd):
        groups.append(dict(params=model.gsd.parameters(), lr=cfg['modulation_lr']))
    optimizer = torch.optim.AdamW(groups, betas=tuple(cfg['betas']), eps=cfg['eps'], weight_decay=cfg['weight_decay'])
    scaler = torch.amp.GradScaler('cuda', enabled=device.type == 'cuda' and cfg['precision'] == 'fp16')
    start, best, history = 0, float('inf'), []
    stopper = EarlyStopping(cfg['early_stopping'], cfg['patience'], cfg['min_delta'])
    latest = folder / 'last.pt'
    if cfg['resume'] and latest.exists():
        ck = torch.load(latest, map_location='cpu', weights_only=False)
        if ck['signature'] != signature:
            raise ValueError('Resume protocol/config/manifest changed; use a fresh output_root')
        model.load_state_dict(ck['model'])
        optimizer.load_state_dict(ck['optimizer'])
        scaler.load_state_dict(ck['scaler'])
        start, best, history = ck['epoch'] + 1, ck['best'], ck['history']
        if 'early_stopping' in ck:
            stopper.state.update(ck['early_stopping'])
        del ck
    if len(train) < cfg['train_batch']:
        raise ValueError('Not enough source-training images for a full logical batch')
    for epoch in range(start, cfg['epochs']):
        if stopper.state['stopped']:
            print('Resuming an early-stopped run: restore best checkpoint and evaluate.')
            break
        model.train()
        running, count, skipped = 0., 0, 0
        for images, labels, _ in tqdm(loader(train, cfg, seed, True, epoch), desc=f'epoch {epoch+1}'):
            bases = estimate_bases(anchor, images, cfg, device, model.selected) if anchor is not None else None
            optimizer.zero_grad(set_to_none=True)
            for a in range(0, len(images), cfg['micro_batch']):
                x, y = images[a:a + cfg['micro_batch']].to(device), labels[a:a + cfg['micro_batch']].to(device)
                with amp(cfg, device):
                    logits = model(x, bases)
                    loss = F.cross_entropy(logits.float(), y, reduction='sum') / len(images)
                if not torch.isfinite(loss):
                    raise FloatingPointError('Nonfinite training loss')
                scaler.scale(loss).backward()
                running += float(loss.detach()) * len(images)
            old_scale = scaler.get_scale()
            scaler.step(optimizer)
            scaler.update()
            skipped += int(scaler.get_scale() < old_scale)
            count += len(images)
        p, _ = predict(model, anchor, val, cfg, seed, device, 'native')
        vloss = float(F.binary_cross_entropy(torch.from_numpy(p).double().clamp(1e-7, 1-1e-7),
                                             torch.tensor(val.label.to_numpy(), dtype=torch.float64)))
        entry = dict(epoch=epoch+1, train_loss=running/count, val_loss=vloss, amp_skipped_steps=skipped,
                     **metric(val.label.to_numpy(), p, cfg['threshold']))
        entry['modulation'] = {l: dict(a=float(g.a), b=float(g.b)) for l, g in model.gsd.items()}
        should_stop = stopper.update(vloss)
        entry['early_stopping'] = dict(stopper.state)
        history.append(entry)
        print(entry)
        if vloss < best:
            best = vloss
            torch.save(dict(model=model.state_dict(), epoch=epoch, signature=signature, config=cfg), folder / 'best.pt')
        torch.save(dict(model=model.state_dict(), optimizer=optimizer.state_dict(), scaler=scaler.state_dict(),
                        epoch=epoch, best=best, history=history, signature=signature,
                        early_stopping=dict(stopper.state)), latest)
        dump(folder / 'history.json', history)
        dump(folder / 'training_status.json', dict(epoch=epoch+1, max_epochs=cfg['epochs'],
             early_stopping=dict(stopper.state), patience=cfg['patience'], min_delta=cfg['min_delta']))
        if should_stop:
            print(f"Early stopping at epoch {epoch+1}: no significant val-loss improvement for {cfg['patience']} epochs.")
            break
    ck = torch.load(folder / 'best.pt', map_location='cpu', weights_only=False)
    model.load_state_dict(ck['model'])
    dump(folder / 'selected_checkpoint.json', dict(epoch=ck['epoch']+1, criterion='minimum source-validation cross entropy', signature=signature))


def summarize(predictions, cfg, out):
    metrics, shifts = [], []
    for (seed, model, case), frame in predictions.groupby(['seed', 'model', 'case']):
        for gen, g in [('pooled', frame)] + list(frame.groupby('generator')):
            metrics.append(dict(seed=seed, model=model, case=case, generator=gen,
                                seen=(gen == cfg['source_generator']), **metric(g.label.to_numpy(), g.prob_fake.to_numpy(), cfg['threshold'])))
    metric_df = pd.DataFrame(metrics)
    metric_df.to_csv(out / 'metrics.csv', index=False)
    per_gen = metric_df[metric_df.generator != 'pooled']
    for protocol, g in [('all', per_gen), ('unseen', per_gen[per_gen.generator != cfg['source_generator']])]:
        g.groupby(['seed', 'model', 'case'])[['auc', 'balanced_accuracy', 'fake_recall', 'real_recall', 'accuracy']].mean().to_csv(out / f'macro_{protocol}.csv')
    for (seed, model), group in predictions.groupby(['seed', 'model']):
        base = group[group.case == 'native'].set_index('sample_id')
        for case, changed in group[group.case != 'native'].groupby('case'):
            changed = changed.set_index('sample_id')
            if set(changed.index) != set(base.index):
                raise ValueError('Paired cohorts differ')
            changed = changed.loc[base.index]
            if not np.array_equal(base.batch_id, changed.batch_id):
                raise ValueError('GSD logical batch membership changed')
            delta = changed.prob_fake - base.prob_fake
            paired = base[['generator', 'label']].copy()
            paired['delta'] = delta
            paired['abs_delta'] = delta.abs()
            paired['flip'] = (changed.prob_fake >= cfg['threshold']) != (base.prob_fake >= cfg['threshold'])
            paired['native_prob'] = base.prob_fake
            paired['changed_prob'] = changed.prob_fake
            paired.reset_index().to_csv(out / f'paired_seed{seed}_{model}_{case}.csv', index=False)
            for (gen, label), g in paired.groupby(['generator', 'label']):
                shifts.append(dict(seed=seed, model=model, case=case, generator=gen, label=label,
                                   mean_delta=g.delta.mean(), mean_abs_delta=g.abs_delta.mean(), flip_rate=g.flip.mean(), n=len(g)))
    pd.DataFrame(shifts).to_csv(out / 'paired_summary.csv', index=False)
    comparison = per_gen.pivot(index=['seed', 'case', 'generator'], columns='model', values=['auc', 'balanced_accuracy', 'fake_recall'])
    comparison.to_csv(out / 'model_comparison.csv')
    if {'clip_ft', 'clip_gsd'} <= set(predictions.model):
        for name in ('auc', 'balanced_accuracy', 'fake_recall'):
            comparison[(name, 'gsd_minus_ft')] = comparison[(name, 'clip_gsd')] - comparison[(name, 'clip_ft')]
        comparison.to_csv(out / 'model_comparison.csv')
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(10, 5))
    plot = per_gen[per_gen.generator != cfg['source_generator']].groupby(['model', 'case']).balanced_accuracy.mean().unstack(0)
    plot.plot.bar(ax=ax)
    ax.set_ylabel('Macro balanced accuracy (unseen generators)')
    ax.set_ylim(0, 1)
    fig.tight_layout()
    fig.savefig(out / 'unseen_balanced_accuracy.png', dpi=160)
    plt.close(fig)
    return metric_df


def run(config=None):
    cfg = {**DEFAULT_CONFIG, **(config or {})}
    if cfg['smoke']:
        cfg.update(epochs=1, train_batch=8, test_batch=8, k=4, strict_eight_generators=False)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    if device.type != 'cuda':
        raise RuntimeError('Use Kaggle GPU; run local unit tests for CPU validation')
    if cfg['precision'] == 'bf16' and not torch.cuda.is_bf16_supported():
        raise ValueError('This GPU does not support BF16; use fp16 on T4/P100')
    if min(cfg['epochs'], cfg['micro_batch'], cfg['anchor_micro_batch'], cfg['train_batch'], cfg['test_batch']) < 1:
        raise ValueError('Batch sizes and epoch count must be positive')
    out = Path(cfg['output_root'])
    out.mkdir(parents=True, exist_ok=True)
    dump(out / 'config.json', cfg)
    import transformers
    dump(out / 'environment.json', dict(torch=torch.__version__, transformers=transformers.__version__,
         pillow=PIL.__version__, python=platform.python_version(), gpu=torch.cuda.get_device_name(0),
         use_devices=1, paper='https://arxiv.org/html/2603.09242v2', torch_threads=torch.get_num_threads()))
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    df = scan_dataset(cfg, out)
    predictions = []
    for seed in cfg['seeds']:
        split_out = out / f'seed{seed}'
        split_out.mkdir(exist_ok=True)
        train, val, tests = make_splits(df, cfg, seed, split_out)
        cfg_hash = digest({**cfg, 'train_files': train.file_hash.tolist(), 'val_files': val.file_hash.tolist(),
                           'test_files': tests.file_hash.tolist()})
        for model_name in cfg['models']:
            if model_name not in ('clip_ft', 'clip_gsd'):
                raise ValueError(model_name)
            seed_all(seed)  # Same backbone AND randomly initialized classifier across the two models.
            folder = split_out / model_name
            folder.mkdir(exist_ok=True)
            model, anchor = build_models(cfg, device, model_name == 'clip_gsd')
            dump(folder / 'architecture.json', dict(backbone=str(model.vision.config),
                 trainable_parameters=sum(p.numel() for p in model.parameters() if p.requires_grad),
                 gsd_pre_block_indices=list(model.selected) if anchor else [],
                 initialization_head_sha=digest(model.head.weight.detach().cpu().tolist()),
                 pooling=cfg['pooling'], batch_basis='entire logical batch, independent of microbatch'))
            train_model(model, anchor, train, val, cfg, seed, device, folder, digest([cfg_hash, model_name]))
            for case in processing_cases(cfg):
                p, batches = predict(model, anchor, tests, cfg, seed, device, case)
                frame = tests[['sample_id', 'generator', 'label', 'pixel_hash']].copy()
                frame = frame.assign(seed=seed, model=model_name, case=case, prob_fake=p, batch_id=batches)
                frame.to_csv(folder / f'predictions_{case}.csv', index=False)
                predictions.append(frame)
            del model, anchor
            gc.collect()
            torch.cuda.empty_cache()
    predictions = pd.concat(predictions, ignore_index=True)
    predictions.to_csv(out / 'predictions.csv', index=False)
    metrics = summarize(predictions, cfg, out)
    dump(out / 'completion.json', dict(status='complete', smoke=cfg['smoke'], config_hash=digest(cfg)))
    print(metrics[metrics.generator == 'pooled'].to_string(index=False))
    return metrics


"""Native-resolution, raw-only All7 training. Also appended to the base runtime in the notebook."""
if 'DEFAULT_CONFIG' not in globals():
    from pathlib import Path
    _base_file = Path(__file__).with_name('runtime.py')
    exec(compile(_base_file.read_text(encoding='utf-8'), str(_base_file), 'exec'), globals())

import time

GENS7 = ['sd15', 'adm', 'biggan', 'glide', 'midjourney', 'wukong', 'vqdm']
DEFAULT_CONFIG = dict(DEFAULT_CONFIG, output_root='/kaggle/working/gsd_raw_all7_native_v1',
    source_generator='all7', train_generators=GENS7, expected_generators=GENS7,
    train_batch=8, test_batch=8, micro_batch=1, anchor_micro_batch=1, num_workers=0,
    jpeg_probability=0., blur_probability=0., horizontal_flip_probability=0.,
    input_mode='native_rgb_edge_pad14', basis_method='gram_eigh', attention='sdpa',
    eval_cases=['native'], scan_log_every=500)

_base_scan = scan_dataset

def scan_dataset(cfg, out):
    print('[scan] Reading all seven train/val folders; RGB decoding and hashing. No resize.', flush=True)
    start = time.monotonic()
    frame = _base_scan(cfg, out)
    frame.groupby(['split','generator','label']).size().rename('n').to_csv(out/'dataset_counts.csv')
    print(f'[scan] {len(frame):,} images audited in {time.monotonic()-start:.1f}s', flush=True)
    return frame


def make_splits(df, cfg, seed, out):
    source = df[(df.split=='train') & df.generator.isin(cfg['train_generators'])].copy()
    # Assign each decoded-pixel group ONCE, even when it occurs in several generators.
    unique = source.drop_duplicates('pixel_hash')
    strata = unique.generator + ':' + unique.label.astype(str)
    a, b = train_test_split(unique, test_size=cfg['validation_fraction'],
                          stratify=strata, random_state=seed)
    def select(frame, limit):
        groups = [g.drop_duplicates('pixel_hash') for _,g in frame.groupby('generator',sort=True)]
        if set(frame.generator)!=set(cfg['train_generators']):
            raise ValueError('Every configured train generator must have a nonempty cohort')
        n = min(int(g.label.value_counts().reindex([0,1],fill_value=0).min()) for g in groups)
        if limit is not None:
            n = min(n, limit)
        if n < 1:
            raise ValueError('A generator lacks real/fake images after hash-group splitting')
        return pd.concat([balanced(g, seed, n) for g in groups],ignore_index=True)
    tr = select(source[source.pixel_hash.isin(a.pixel_hash)], cfg['smoke_train_per_class'] if cfg['smoke'] else None)
    va = select(source[source.pixel_hash.isin(b.pixel_hash)], cfg['smoke_eval_per_class'] if cfg['smoke'] else None)
    tests = df[df.split=='test'].copy()
    removed = tests[tests.pixel_hash.isin(source.pixel_hash)]
    removed.to_csv(out/'excluded_test_overlap.csv',index=False)
    tests = tests[~tests.pixel_hash.isin(source.pixel_hash)].drop_duplicates(['generator','pixel_hash'])
    tests = pd.concat([balanced(g,seed,cfg['smoke_eval_per_class'] if cfg['smoke'] else None)
                       for _,g in tests.groupby('generator',sort=True)],ignore_index=True)
    def ordered(f):
        return f.assign(_order=f.sample_id.map(lambda s:digest([seed,s]))).sort_values('_order').drop(columns='_order').reset_index(drop=True)
    tr,va,tests=map(ordered,(tr,va,tests))
    assert not set(tr.pixel_hash)&set(va.pixel_hash)
    assert not (set(tr.pixel_hash)|set(va.pixel_hash))&set(tests.pixel_hash)
    for name,frame in [('train',tr),('validation',va),('test',tests)]:
        frame.to_csv(out/f'{name}_manifest.csv',index=False)
        frame.groupby(['generator','label']).size().rename('n').to_csv(out/f'{name}_counts.csv')
    dump(out/'split_summary.json',dict(train=len(tr),validation=len(va),test=len(tests),
        test_overlap_removed=len(removed),selection='All7 raw; unique pixel groups split before per-generator/class balancing',
        train_hash=digest(tr.sample_id.tolist()),validation_hash=digest(va.sample_id.tolist()),test_hash=digest(tests.sample_id.tolist())))
    print(f'[split] train={len(tr):,}, validation={len(va):,}, test={len(tests):,}; all 7 seen',flush=True)
    return tr,va,tests


def processing_cases(cfg):
    return cfg['eval_cases']


def input_geometry(width, height, cfg):
    """Source-pixel crop coordinates and exact model dimensions, without resampling."""
    cap = cfg['crop_max_side'] if cfg['input_mode']=='center_crop_max_edge_pad14' else None
    cw = min(width, cap) if cap else width
    ch = min(height, cap) if cap else height
    left, top = (width-cw)//2, (height-ch)//2
    pw, ph = ((cw+13)//14)*14, ((ch+13)//14)*14
    return dict(crop_left=left,crop_top=top,crop_width=cw,crop_height=ch,
                input_width=pw,input_height=ph,grid_w=pw//14,grid_h=ph//14,
                tokens=(pw//14)*(ph//14)+1)


def protocol_id(cfg):
    if cfg['input_mode']=='center_crop_max_edge_pad14':
        return f"raw_all7_center_crop{cfg['crop_max_side']}_v1"
    return 'raw_all7_native_v1'


class Images(Dataset):
    def __init__(self,frame,cfg,seed,train=False,epoch=0,case='native'):
        self.rows=frame.to_dict('records'); self.cfg=cfg; self.case=case; self.train=train
        if train and case!='native':
            raise ValueError('Training must use native raw images')

    def __len__(self):
        return len(self.rows)

    def __getitem__(self,i):
        row=self.rows[i]
        with Image.open(row['path']) as im:
            image=im.convert('RGB')
        if not self.train:
            if self.case.startswith('fake_jpeg'):
                if int(row['label'])==1:
                    image=jpeg(image,int(self.case[len('fake_jpeg'):]),self.cfg['jpeg_subsampling'])
            elif self.case!='native':
                image=process(image,self.case,self.cfg)
        geom=input_geometry(*image.size,self.cfg)
        image=image.crop((geom['crop_left'],geom['crop_top'],
                          geom['crop_left']+geom['crop_width'],geom['crop_top']+geom['crop_height']))
        x=torch.from_numpy(np.asarray(image,dtype=np.float32).copy()).permute(2,0,1)/255.
        # Preserve pixels inside the selected crop. No resize or upscaling of smaller images.
        h,w=x.shape[-2:]
        x=F.pad(x.unsqueeze(0),(0,(-w)%14,0,(-h)%14),mode='replicate').squeeze(0)
        mean=x.new_tensor([.48145466,.4578275,.40821073])[:,None,None]
        std=x.new_tensor([.26862954,.26130258,.27577711])[:,None,None]
        return (x-mean)/std,int(row['label']),i


def native_collate(items):
    x,y,i=zip(*items)
    return list(x),torch.tensor(y,dtype=torch.long),torch.tensor(i,dtype=torch.long)


def loader(frame,cfg,seed,train=False,epoch=0,case='native'):
    return DataLoader(Images(frame,cfg,seed,train,epoch,case),
        batch_size=cfg['train_batch'] if train else cfg['test_batch'], shuffle=train,
        generator=torch.Generator().manual_seed(seed+epoch),num_workers=cfg['num_workers'],
        collate_fn=native_collate,pin_memory=torch.cuda.is_available(),drop_last=False)


def image_chunks(images,limit):
    # Keep membership and order of the mixed logical batch, regardless of shape or label.
    start=0
    while start<len(images):
        end=start+1
        while end<min(start+limit,len(images)) and images[end].shape==images[start].shape:
            end+=1
        yield start,torch.stack(images[start:end])
        start=end


class NativeDetector(Detector):
    def forward(self,x,bases=None):
        h=self.vision.pre_layrnorm(self.vision.embeddings(x,interpolate_pos_encoding=True))
        for l,block in enumerate(self.vision.encoder.layers):
            if str(l) in self.gsd:
                h=self.gsd[str(l)](h,bases[l])
            def apply(t,layer=block):
                return layer(t,attention_mask=None,causal_attention_mask=None)[0]
            h=checkpoint(apply,h,use_reentrant=False) if self.training and self.cfg['gradient_checkpointing'] else apply(h)
        return self.head(self.vision.post_layernorm(h)[:,1:].mean(1))


@torch.no_grad()
def estimate_bases(anchor,images,cfg,device,selected):
    # M.T @ M has the same principal right-singular subspace as uncentered M.
    # Stream ALL tokens: no token sampling and no BN x d concatenation on GPU.
    grams={l:torch.zeros(anchor.config.hidden_size,anchor.config.hidden_size,device=device,dtype=torch.float32) for l in selected}
    tokens=0
    for _,x in image_chunks(images,cfg['anchor_micro_batch']):
        with amp(cfg,device):
            h=anchor.pre_layrnorm(anchor.embeddings(x.to(device),interpolate_pos_encoding=True))
            tokens+=h.shape[0]*(h.shape[1]-1)
            for l,layer in enumerate(anchor.encoder.layers):
                if l in grams:
                    with torch.autocast(device.type,enabled=False):
                        m=h[:,1:].flatten(0,1).float()
                        grams[l].add_(m.T@m)
                h=layer(h,attention_mask=None,causal_attention_mask=None)[0]
    if cfg['k']>min(tokens,anchor.config.hidden_size):
        raise ValueError(f"k={cfg['k']} exceeds token/feature dimensions of this logical batch ({tokens})")
    bases={}
    with torch.autocast(device.type,enabled=False):
        for l,g in grams.items():
            _,v=torch.linalg.eigh((g+g.T)*(0.5/tokens))
            bases[l]=v[:,-cfg['k']:].contiguous().detach()
    return bases


def build_models(cfg,device,gsd):
    pretrained=CLIPVisionModel.from_pretrained(cfg['model_id'],local_files_only=cfg['local_files_only'],attn_implementation=cfg['attention'])
    v=pretrained.vision_model
    if (v.config.hidden_size,v.config.patch_size)!=(1024,14):
        raise ValueError('Native All7 protocol requires CLIP ViT-L/14')
    model=NativeDetector(v,cfg,gsd).to(device)
    anchor=copy.deepcopy(v).requires_grad_(False).eval().to(device) if gsd else None
    if anchor is not None and device.type=='cuda' and cfg['precision']!='fp32':
        anchor.to(dtype=torch.float16 if cfg['precision']=='fp16' else torch.bfloat16)
    return model,anchor


@torch.no_grad()
def predict(model,anchor,frame,cfg,seed,device,case):
    model.eval(); result=np.full(len(frame),np.nan); batches=np.full(len(frame),-1)
    for bid,(images,_,indices) in enumerate(loader(frame,cfg,seed,case=case)):
        bases=estimate_bases(anchor,images,cfg,device,model.selected) if anchor is not None else None
        for start,x in image_chunks(images,cfg['micro_batch']):
            with amp(cfg,device):
                logits=model(x.to(device),bases)
            ix=indices[start:start+len(x)].numpy()
            result[ix]=logits.float().softmax(-1)[:,1].cpu().numpy(); batches[ix]=bid
        if bid%25==0:
            print(f'[{case}] batch {bid+1}/{math.ceil(len(frame)/cfg["test_batch"])}',flush=True)
    if not np.isfinite(result).all():
        raise ValueError('Missing/nonfinite prediction')
    return result,batches


def save_checkpoint(path,state):
    temp=path.with_suffix('.tmp')
    torch.save(state,temp)
    temp.replace(path)


def train_model(model,anchor,train,val,cfg,seed,device,folder,signature):
    groups=[dict(params=model.vision.parameters(),lr=cfg['lr']),dict(params=model.head.parameters(),lr=cfg['head_lr'])]
    if len(model.gsd):
        groups.append(dict(params=model.gsd.parameters(),lr=cfg['modulation_lr']))
    optimizer=torch.optim.AdamW(groups,betas=tuple(cfg['betas']),eps=cfg['eps'],weight_decay=cfg['weight_decay'])
    scaler=torch.amp.GradScaler('cuda',enabled=device.type=='cuda' and cfg['precision']=='fp16')
    stopper=EarlyStopping(cfg['early_stopping'],cfg['patience'],cfg['min_delta'])
    latest=folder/'last.pt'; start=0; best=float('inf'); history=[]
    if cfg['resume'] and latest.exists():
        ck=torch.load(latest,map_location='cpu',weights_only=False)
        if ck['signature']!=signature:
            raise ValueError('Resume config/manifests changed; use a new output_root')
        model.load_state_dict(ck['model']); optimizer.load_state_dict(ck['optimizer']); scaler.load_state_dict(ck['scaler'])
        stopper.state.update(ck['early_stopping']); start=ck['epoch']+1; best=ck['best']; history=ck['history']
        del ck
        print(f'[resume] next epoch {start+1}, stopped={stopper.state["stopped"]}',flush=True)
    for epoch in range(start,cfg['epochs']):
        if stopper.state['stopped']:
            break
        model.train(); running=0.; count=0; skipped=0; tick=time.monotonic()
        for bid,(images,labels,_) in enumerate(loader(train,cfg,seed,True,epoch)):
            bases=estimate_bases(anchor,images,cfg,device,model.selected) if anchor is not None else None
            optimizer.zero_grad(set_to_none=True)
            for a,x in image_chunks(images,cfg['micro_batch']):
                with amp(cfg,device):
                    logits=model(x.to(device),bases)
                    loss=F.cross_entropy(logits.float(),labels[a:a+len(x)].to(device),reduction='sum')/len(images)
                if not torch.isfinite(loss):
                    raise FloatingPointError('Nonfinite loss')
                scaler.scale(loss).backward(); running+=float(loss.detach())*len(images)
            old=scaler.get_scale(); scaler.step(optimizer); scaler.update(); skipped+=int(scaler.get_scale()<old); count+=len(images)
            if bid%10==0:
                print(f'[epoch {epoch+1}] batch {bid+1}/{math.ceil(len(train)/cfg["train_batch"])}; loss={running/count:.5f}',flush=True)
        p,_=predict(model,anchor,val,cfg,seed,device,'native')
        vloss=float(F.binary_cross_entropy(torch.from_numpy(p).double().clamp(1e-7,1-1e-7),torch.tensor(val.label.to_numpy(),dtype=torch.float64)))
        stop=stopper.update(vloss)
        entry=dict(epoch=epoch+1,train_loss=running/count,val_loss=vloss,seconds=time.monotonic()-tick,
            amp_skipped_steps=skipped,train_images=count,**metric(val.label.to_numpy(),p,cfg['threshold']),
            modulation={l:dict(a=float(g.a.detach()),b=float(g.b.detach())) for l,g in model.gsd.items()},early_stopping=dict(stopper.state))
        history.append(entry); print(entry,flush=True)
        if vloss<best:
            best=vloss; save_checkpoint(folder/'best.pt',dict(model=model.state_dict(),epoch=epoch,signature=signature,config=cfg))
        save_checkpoint(latest,dict(model=model.state_dict(),optimizer=optimizer.state_dict(),scaler=scaler.state_dict(),
            epoch=epoch,best=best,history=history,signature=signature,config=cfg,early_stopping=dict(stopper.state)))
        dump(folder/'history.json',history)
        dump(folder/'training_status.json',dict(epoch=epoch+1,max_epochs=cfg['epochs'],early_stopping=dict(stopper.state)))
        if stop:
            print('[early stopping] Restore best checkpoint for evaluation.',flush=True)
            break
    ck=torch.load(folder/'best.pt',map_location='cpu',weights_only=False); model.load_state_dict(ck['model'])
    dump(folder/'selected_checkpoint.json',dict(epoch=ck['epoch']+1,criterion='minimum All7 raw validation cross entropy',signature=signature))


def summarize(predictions,cfg,out):
    rows=[]
    for (seed,model,case),f in predictions.groupby(['seed','model','case']):
        for gen,g in [('pooled',f),*f.groupby('generator')]:
            rows.append(dict(seed=seed,model=model,case=case,generator=gen,
                seen=gen in cfg['train_generators'],**metric(g.label.to_numpy(),g.prob_fake.to_numpy(),cfg['threshold'])))
    result=pd.DataFrame(rows); result.to_csv(out/'metrics.csv',index=False)
    result[result.generator!='pooled'].groupby(['seed','model','case'])[['auc','balanced_accuracy','fake_recall','real_recall','accuracy']].mean().to_csv(out/'macro_all.csv')
    return result


def validate_raw_config(cfg):
    if set(cfg['train_generators'])!=set(GENS7) or set(cfg['expected_generators'])!=set(GENS7):
        raise ValueError('This protocol requires exactly all seven Tiny generators')
    if any(cfg[k]!=0 for k in ('jpeg_probability','blur_probability','horizontal_flip_probability')):
        raise ValueError('Raw-only training forbids JPEG/blur/flip augmentation')
    if cfg['input_mode'] not in ('native_rgb_edge_pad14','center_crop_max_edge_pad14') or cfg['basis_method']!='gram_eigh':
        raise ValueError('Unsupported native-input protocol')
    if cfg['input_mode']=='center_crop_max_edge_pad14':
        cap=cfg.get('crop_max_side')
        if isinstance(cap,bool) or not isinstance(cap,int) or cap<14 or cap%14:
            raise ValueError('crop_max_side must be a positive integer multiple of 14')
    if cfg['attention']!='sdpa' or cfg['pooling']!='patch_mean_post_ln':
        raise ValueError('Native protocol requires SDPA and patch_mean_post_ln')
    if min(cfg[k] for k in ('train_batch','test_batch','micro_batch','anchor_micro_batch','epochs','k'))<1:
        raise ValueError('Positive batches, epochs and k required')
    if cfg['precision'] not in ('fp16','bf16','fp32'):
        raise ValueError('precision must be fp16, bf16 or fp32')
    if not cfg['eval_cases'] or 'native' not in cfg['eval_cases']:
        raise ValueError('Evaluation must include native')
    for case in cfg['eval_cases']:
        if case not in ('native','size') and not any(case==prefix+str(q) for prefix in ('jpeg','fake_jpeg','size_jpeg') for q in range(1,101)):
            raise ValueError(f'Unknown evaluation case: {case}')


def run(config=None):
    cfg={**DEFAULT_CONFIG,**(config or {})}; validate_raw_config(cfg)
    if cfg['smoke']:
        cfg.update(epochs=1,train_batch=2,test_batch=2,k=4)
    device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    if device.type!='cuda':
        raise RuntimeError('Use Kaggle GPU; CPU unit tests cover native input separately')
    if cfg['precision']=='bf16' and not torch.cuda.is_bf16_supported():
        raise ValueError('GPU does not support bf16; use fp16 on T4')
    out=Path(cfg['output_root']); out.mkdir(parents=True,exist_ok=True)
    if (out/'config.json').exists():
        previous=json.loads((out/'config.json').read_text(encoding='utf-8'))
        if previous!=cfg:
            raise ValueError('Existing output config differs; choose a new output_root before running')
    dump(out/'config.json',cfg)
    dump(out/'environment.json',dict(torch=torch.__version__,pillow=PIL.__version__,python=platform.python_version(),
        gpu=torch.cuda.get_device_name(0),use_devices=1,protocol=protocol_id(cfg)))
    torch.backends.cuda.matmul.allow_tf32=False; torch.backends.cudnn.allow_tf32=False
    df=scan_dataset(cfg,out)
    census=df[['sample_id','split','generator','label','width','height']].copy()
    geometry=pd.DataFrame([input_geometry(int(w),int(h),cfg) for w,h in zip(census.width,census.height)])
    census=pd.concat([census.reset_index(drop=True),geometry],axis=1)
    census.to_csv(out/'token_census.csv',index=False)
    print(f'[{protocol_id(cfg)}] tokens min/median/max: {census.tokens.min()}/{census.tokens.median()}/{census.tokens.max()}. No pixel resampling.',flush=True)
    results=[]
    for seed in cfg['seeds']:
        split_out=out/f'seed{seed}'; split_out.mkdir(exist_ok=True)
        train,val,tests=make_splits(df,cfg,seed,split_out)
        signature=digest(dict(config=cfg,train=train.file_hash.tolist(),val=val.file_hash.tolist(),test=tests.file_hash.tolist(),protocol=protocol_id(cfg)))
        for name in cfg['models']:
            if name not in ('clip_ft','clip_gsd'):
                raise ValueError(name)
            seed_all(seed); folder=split_out/name; folder.mkdir(exist_ok=True)
            print(f'[model] Loading {name}; native position interpolation in BOTH branches.',flush=True)
            model,anchor=build_models(cfg,device,name=='clip_gsd')
            dump(folder/'architecture.json',dict(backbone=str(model.vision.config),input_mode=cfg['input_mode'],
                basis=cfg['basis_method'],attention=cfg['attention'],all_generators_seen=True,patch_size=14,
                protocol=protocol_id(cfg),crop_max_side=cfg.get('crop_max_side'),
                padding='right/bottom edge replication to multiple of 14; no resize',
                trainable_parameters=sum(p.numel() for p in model.parameters() if p.requires_grad)))
            try:
                train_model(model,anchor,train,val,cfg,seed,device,folder,signature)
                for case in processing_cases(cfg):
                    p,batches=predict(model,anchor,tests,cfg,seed,device,case)
                    f=tests[['sample_id','generator','label','pixel_hash','width','height']].assign(seed=seed,model=name,case=case,prob_fake=p,batch_id=batches)
                    f.to_csv(folder/f'predictions_{case}.csv',index=False); results.append(f)
            except torch.cuda.OutOfMemoryError as exc:
                raise RuntimeError('CUDA OOM under the configured input protocol. No automatic resize, skip or batch reduction. Use more VRAM or explicitly start a smaller-input/batch protocol; last.pt retains completed epochs.') from exc
            del model,anchor; gc.collect(); torch.cuda.empty_cache()
    predictions=pd.concat(results,ignore_index=True); predictions.to_csv(out/'predictions.csv',index=False)
    metrics=summarize(predictions,cfg,out)
    dump(out/'completion.json',dict(status='complete',smoke=cfg['smoke'],protocol=protocol_id(cfg),config_hash=digest(cfg)))
    print(metrics[metrics.generator=='pooled'].to_string(index=False),flush=True)
    return metrics


"""Trusted audit-manifest reuse. Loaded after runtime.py; never decode/hash image content.

The caller deliberately trusts the previous image audit and the same dataset version.
Filesystem stat checks cannot detect content changes that keep the same byte length.
"""
from pathlib import Path, PurePosixPath
import re

MANIFEST_PARTITIONS = ('train', 'validation', 'test')
MANIFEST_COLUMNS = {
    'sample_id', 'relative_path', 'generator', 'split', 'label', 'width', 'height',
    'file_hash', 'pixel_hash', 'file_bytes',
}


def find_manifest_root(cfg):
    def complete(root):
        return all((root / f'seed{seed}' / f'{part}_manifest.csv').is_file()
                   for seed in cfg['seeds'] for part in MANIFEST_PARTITIONS)

    if cfg['manifest_root']:
        root = Path(cfg['manifest_root']).resolve()
        if re.fullmatch(r'seed\d+', root.name):
            root = root.parent
        if not complete(root):
            raise FileNotFoundError(f'Missing saved train/validation/test manifests under {root}/seed<seed>. '
                                    'Attach the previous All7 output; no full-scan fallback is enabled.')
        return root
    search = Path(cfg['input_root'])
    roots = sorted({p.parent.parent.resolve() for p in search.rglob('train_manifest.csv')
                    if p.parent.name == f'seed{cfg["seeds"][0]}' and complete(p.parent.parent)})
    if len(roots) != 1:
        raise ValueError(f'Set manifest_root explicitly. Complete All7-manifest candidates: {[str(p) for p in roots]}. '
                         'Need seed42/train_manifest.csv, validation_manifest.csv and test_manifest.csv; '
                         'weights alone are not enough. No image scan was started.')
    return roots[0]


def validate_manifest_frame(frame, cfg, partition):
    missing = MANIFEST_COLUMNS - set(frame)
    if missing or frame.empty:
        raise ValueError(f'Invalid {partition} manifest: missing columns={sorted(missing)} or empty cohort')
    frame = frame.copy()
    if frame[list(MANIFEST_COLUMNS)].isna().any().any():
        raise ValueError(f'Null required metadata in {partition} manifest')
    if frame.sample_id.duplicated().any() or frame.relative_path.duplicated().any():
        raise ValueError(f'Duplicate IDs/paths in {partition} manifest')
    for column in ('sample_id', 'file_hash', 'pixel_hash'):
        if not frame[column].astype(str).str.fullmatch(r'[0-9a-f]{64}').all():
            raise ValueError(f'Invalid saved {column} in {partition} manifest')
    for column in ('label', 'width', 'height', 'file_bytes'):
        values = pd.to_numeric(frame[column], errors='coerce')
        if not np.isfinite(values).all() or (values != np.floor(values)).any():
            raise ValueError(f'Invalid integer {column} in {partition} manifest')
        frame[column] = values.astype('int64')
    if not frame.label.isin([0, 1]).all() or (frame[['width', 'height', 'file_bytes']] <= 0).any().any():
        raise ValueError(f'Invalid labels/dimensions/byte lengths in {partition} manifest')
    source_split = 'test' if partition == 'test' else 'train'
    if set(frame.split) != {source_split}:
        raise ValueError(f'{partition} manifest must originate only from source {source_split} folders')
    counts = frame.groupby(['generator', 'label']).size()
    expected = {(g, y) for g in cfg['train_generators'] for y in (0, 1)}
    if set(counts.index) != expected or counts.nunique() != 1:
        raise ValueError(f'{partition} manifest must contain balanced real/fake cohorts for all seven generators')
    if frame.duplicated(['generator', 'pixel_hash']).any():
        raise ValueError(f'Duplicate pixels within a generator in {partition} manifest')
    label_names = {'nature': 0, 'real': 0, '0_real': 0, 'ai': 1, 'fake': 1, '1_fake': 1}
    split_names = {'train': {'train', 'training'}, 'test': {'val', 'valid', 'validation', 'test'}}
    for row in frame.itertuples():
        rel = str(row.relative_path)
        parts = PurePosixPath(rel).parts
        if (not parts or len(parts) < 4 or PurePosixPath(rel).is_absolute() or
                '\\' in rel or ':' in rel or '..' in parts or rel != PurePosixPath(rel).as_posix()):
            raise ValueError(f'Unsafe relative image path: {rel}')
        if (canonical_generator(parts[0]) != row.generator or parts[1] not in split_names[source_split] or
                label_names.get(parts[2].lower()) != row.label):
            raise ValueError(f'Path/generator/split/label mismatch: {rel}')
        if row.sample_id != hashlib.sha256(rel.encode()).hexdigest():
            raise ValueError(f'Saved sample_id does not match relative path: {rel}')
    return frame


def read_manifest_cohorts(root, cfg, seed):
    folder = root / f'seed{seed}'
    summary_path = folder / 'split_summary.json'
    summary = json.loads(summary_path.read_text(encoding='utf-8')) if summary_path.is_file() else None
    frames, sources = {}, {}
    for partition in MANIFEST_PARTITIONS:
        path = folder / f'{partition}_manifest.csv'
        data = path.read_bytes()  # Hash only the small CSV, NOT any source image.
        frame = pd.read_csv(io.BytesIO(data), dtype={k: str for k in
            ('sample_id', 'relative_path', 'generator', 'split', 'file_hash', 'pixel_hash')})
        frame = validate_manifest_frame(frame, cfg, partition)
        if summary is not None:
            if (summary.get(partition) != len(frame) or
                    summary.get(partition + '_hash') != digest(frame.sample_id.tolist())):
                raise ValueError(f'{path}: ordered cohort does not match saved split_summary.json')
        frames[partition] = frame
        sources[partition] = dict(path=str(path), csv_sha256=hashlib.sha256(data).hexdigest(),
                                  count=len(frame), ordered_ids_hash=digest(frame.sample_id.tolist()))
    for a, b in (('train', 'validation'), ('train', 'test'), ('validation', 'test')):
        if set(frames[a].sample_id) & set(frames[b].sample_id):
            raise ValueError(f'Saved IDs overlap across {a}/{b}')
        if set(frames[a].pixel_hash) & set(frames[b].pixel_hash):
            raise ValueError(f'Saved pixel hashes overlap across {a}/{b}')
    joined = pd.concat(list(frames.values()), ignore_index=True)
    if (joined.groupby('pixel_hash').label.nunique() > 1).any():
        raise ValueError('Conflicting saved labels for identical pixel hashes')
    return frames, sources


def resolve_manifest_dataset(frame, cfg):
    first = frame.relative_path.iloc[0]
    if cfg['dataset_root']:
        candidates = [Path(cfg['dataset_root'])]
    else:
        search = Path(cfg['input_root'])
        candidates = [search / 'tiny-genimage', search / 'datasets/yangsangtai/tiny-genimage']
        generator_dir = PurePosixPath(first).parts[0]
        print('[manifest] Locating dataset root using directory metadata only.', flush=True)
        candidates.extend(p.parent for p in search.rglob(generator_dir) if p.is_dir())
    matches = sorted({p.resolve() for p in candidates if (p / first).is_file()})
    if len(matches) != 1:
        raise ValueError(f'Set dataset_root to the directory containing generator folders. Candidates: '
                         f'{[str(p) for p in matches]}. No RGB/hash scan was started.')
    root = matches[0]
    resolved = frame.copy()
    paths = []
    for index, row in enumerate(resolved.itertuples()):
        path = (root / row.relative_path).resolve()
        try:
            path.relative_to(root)
        except ValueError:
            raise ValueError(f'Image path escapes dataset root: {row.relative_path}')
        if not path.is_file() or path.stat().st_size != row.file_bytes:
            raise ValueError(f'Missing or changed file (path/size check): {path}. '
                             'Use the same dataset version as the saved audit.')
        paths.append(str(path))
        if (index + 1) % 5000 == 0:
            print(f'[manifest] {index + 1:,}/{len(frame):,} file paths/sizes checked; no image decode/hash.', flush=True)
    resolved['path'] = paths  # Ignore stale absolute /kaggle/input paths from the previous run.
    return resolved, root


def load_manifest_index(cfg, out):
    root = find_manifest_root(cfg)
    source_config_path = root / 'config.json'
    if source_config_path.is_file():
        source = json.loads(source_config_path.read_text(encoding='utf-8'))
        if source.get('smoke') and not cfg['smoke']:
            raise ValueError('Cannot use smoke-only source manifests for a full training run')
        if set(source.get('train_generators', [])) != set(cfg['train_generators']):
            raise ValueError('Source config is not an All7 experiment')
        if source.get('validation_fraction') != cfg['validation_fraction']:
            raise ValueError('validation_fraction differs from saved split; reused manifests are never re-split')
    all_frames, cohorts, sources = [], {}, {}
    for seed in cfg['seeds']:
        frames, provenance = read_manifest_cohorts(root, cfg, seed)
        all_frames.extend(frames.values())
        cohorts[str(seed)] = {part: f.sample_id.tolist() for part, f in frames.items()}
        sources[str(seed)] = provenance
    merged = pd.concat(all_frames, ignore_index=True)
    # A sample may occur in several seed experiments; metadata must agree.
    for column in sorted(MANIFEST_COLUMNS - {'sample_id'}):
        if (merged.groupby('sample_id')[column].nunique(dropna=False) > 1).any():
            raise ValueError(f'Conflicting saved {column} across seeds/cohorts')
    merged = merged.drop_duplicates('sample_id', keep='first').reset_index(drop=True)
    merged, dataset = resolve_manifest_dataset(merged, cfg)
    provenance = dict(mode='reuse_manifests', manifest_root=str(root), dataset_root=str(dataset),
                      sources=sources, unique_selected_images=len(merged),
                      checks=['schema', 'ordered_split_summary_if_present', 'saved_pixel_hash_leakage',
                              'label_and_split_path_consistency', 'file_exists', 'file_byte_length'],
                      skipped=['image_decode', 'image_content_hash', 'pixel_hash_recomputation',
                               'discovery_of_unlisted_or_new_images'],
                      trust='Saved audit metadata and unchanged dataset version are trusted. '
                            'Same-size content changes are NOT detected.')
    target = out / 'manifest_reuse.json'
    if target.exists() and json.loads(target.read_text(encoding='utf-8')) != provenance:
        raise ValueError('Manifest/input fingerprint changed; choose a new output_root before resume')
    dump(target, provenance)
    save_csv(out / 'reused_manifest_index.csv', merged)
    merged.attrs['reused_cohorts'] = cohorts
    merged.attrs['manifest_root'] = str(root)
    print(f'[manifest] Reused {len(merged):,} selected images. No RGB decoding or image hashing; no re-split.', flush=True)
    return merged


def reuse_saved_splits(df, cfg, seed, out):
    cohorts = df.attrs.get('reused_cohorts', {}).get(str(seed))
    if cohorts is None:
        raise ValueError('Missing validated saved-cohort index; refusing to re-split or scan implicitly')
    indexed = df.set_index('sample_id', drop=False)
    frames = []
    for partition in MANIFEST_PARTITIONS:
        frame = indexed.loc[cohorts[partition]].reset_index(drop=True)
        frame.attrs = {}
        if cfg['smoke']:
            limit = cfg['smoke_train_per_class'] if partition == 'train' else cfg['smoke_eval_per_class']
            frame = frame[frame.groupby(['generator', 'label']).cumcount() < limit].reset_index(drop=True)
        save_csv(out / f'{partition}_manifest.csv', frame)
        frame.groupby(['generator', 'label']).size().rename('n').to_csv(out / f'{partition}_counts.csv')
        frames.append(frame)
    tr, va, te = frames
    dump(out / 'split_summary.json', dict(train=len(tr), validation=len(va), test=len(te),
         train_hash=digest(tr.sample_id.tolist()), validation_hash=digest(va.sample_id.tolist()),
         test_hash=digest(te.sample_id.tolist()), selection='Reused ordered audited All7 cohorts; no re-split',
         manifest_root=df.attrs['manifest_root'], smoke_subset=bool(cfg['smoke'])))
    print(f'[split/reused] train={len(tr):,}, validation={len(va):,}, test={len(te):,}; original ID order retained.', flush=True)
    return tr, va, te


"""Inference-only evaluation for All7 crop560 checkpoints; resumable complete logical batches."""
if 'NativeDetector' not in globals():
    from pathlib import Path
    _here=Path(__file__).resolve().parent
    exec(compile((_here/'runtime.py').read_text(encoding='utf-8'),str(_here/'runtime.py'),'exec'),globals())
    exec(compile((_here/'runtime_raw_all7.py').read_text(encoding='utf-8'),str(_here/'runtime_raw_all7.py'),'exec'),globals())

from torch.utils.data import Subset
import transformers

EVAL_DEFAULTS=dict(
    input_root='/kaggle/input', checkpoint_path=None, dataset_root=None,
    output_root='/kaggle/working/gsd_all7_crop560_variants', pretrained_model_path=None,
    expected_training_signature='18ea6abe15bbb769490788a8ec5516457b7538ebb3dad1018d4a94948a2a79cc',
    cases=['native','png_roundtrip','fake_jpeg96','all_jpeg96','all_jpeg90','all_jpeg80',
           'resize_square256','resize_square560','blur_sigma1','resize256_jpeg96'],
    micro_batch=1,anchor_micro_batch=1,num_workers=0,save_every_batches=25,log_every_batches=10,
    max_batches=None, native_tolerance=1e-3,verify_file_sizes=True)


def case_spec(name):
    fixed={'native':dict(kind='native'), 'png_roundtrip':dict(kind='png'),
           'blur_sigma1':dict(kind='blur',sigma=1.),'blur_sigma2':dict(kind='blur',sigma=2.),
           'resize256_jpeg96':dict(kind='resize_jpeg',side=256,quality=96),
           'resize_long560':dict(kind='resize_long',side=560)}
    if name in fixed:
        return fixed[name]
    for prefix,kind in [('fake_jpeg','fake_jpeg'),('all_jpeg','all_jpeg'),('resize_square','resize_square')]:
        if name.startswith(prefix) and name[len(prefix):].isdigit():
            value=int(name[len(prefix):])
            if kind in ('fake_jpeg','all_jpeg') and 1<=value<=100:
                return dict(kind=kind,quality=value)
            if kind=='resize_square' and value in (224,256,336,512,560):
                return dict(kind=kind,side=value)
    raise ValueError(f'Unsupported case: {name}')


def transform_variant(image,label,spec,cfg):
    kind=spec['kind']
    if kind=='png':
        buf=io.BytesIO(); image.save(buf,format='PNG'); buf.seek(0)
        with Image.open(buf) as decoded:
            image=decoded.convert('RGB').copy()
    elif kind in ('all_jpeg','fake_jpeg'):
        if kind=='all_jpeg' or label==1:
            image=jpeg(image,spec['quality'],cfg['jpeg_subsampling'])
    elif kind=='blur':
        image=image.filter(ImageFilter.GaussianBlur(spec['sigma']))
    elif kind in ('resize_square','resize_jpeg'):
        image=image.resize((spec['side'],spec['side']),Image.Resampling.BICUBIC)
        if kind=='resize_jpeg':
            image=jpeg(image,spec['quality'],cfg['jpeg_subsampling'])
    elif kind=='resize_long':
        w,h=image.size
        ratio=min(1.,spec['side']/max(w,h))
        image=image.resize((max(1,round(w*ratio)),max(1,round(h*ratio))),Image.Resampling.BICUBIC)
    elif kind!='native':
        raise ValueError(kind)
    return image


class VariantImages(Dataset):
    def __init__(self,frame,cfg,spec):
        self.rows=frame.to_dict('records'); self.cfg=cfg; self.spec=spec

    def __len__(self):
        return len(self.rows)

    def __getitem__(self,index):
        row=self.rows[index]
        with Image.open(row['path']) as im:
            original=im.convert('RGB')
        if original.size!=(int(row['width']),int(row['height'])):
            raise ValueError(f"Image dimensions changed: {row['relative_path']}")
        image=transform_variant(original,int(row['label']),self.spec,self.cfg)
        geom=input_geometry(*image.size,self.cfg)
        image=image.crop((geom['crop_left'],geom['crop_top'],geom['crop_left']+geom['crop_width'],geom['crop_top']+geom['crop_height']))
        x=torch.from_numpy(np.asarray(image,dtype=np.float32).copy()).permute(2,0,1)/255.
        h,w=x.shape[-2:]; x=F.pad(x[None],(0,(-w)%14,0,(-h)%14),mode='replicate')[0]
        x=(x-x.new_tensor([.48145466,.4578275,.40821073])[:,None,None])/x.new_tensor([.26862954,.26130258,.27577711])[:,None,None]
        return x,int(row['label']),index


def file_sha(path):
    hasher=hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda:stream.read(8*1024*1024),b''):
            hasher.update(chunk)
    return hasher.hexdigest()


def find_checkpoint(options):
    if options['checkpoint_path']:
        path=Path(options['checkpoint_path'])
        if not path.is_file():
            raise FileNotFoundError(path)
        return path
    candidates=[]
    for p in Path(options['input_root']).rglob('best.pt'):
        root=p.parents[2]
        if (root/'seed42/test_manifest.csv').is_file() and (root/'config.json').is_file():
            config=json.loads((root/'config.json').read_text(encoding='utf-8'))
            if config.get('input_mode')=='center_crop_max_edge_pad14' and config.get('crop_max_side')==560:
                candidates.append(p)
    if len(candidates)!=1:
        raise ValueError(f'Set checkpoint_path explicitly. Crop560 best.pt candidates: {[str(p) for p in candidates]}')
    return candidates[0]


def resolve_test_files(frame,options):
    required={'sample_id','relative_path','label','generator','width','height','file_hash','pixel_hash','file_bytes'}
    if not required<=set(frame.columns) or frame.sample_id.duplicated().any():
        raise ValueError('Invalid saved test manifest')
    for relative in frame.relative_path:
        if Path(relative).is_absolute() or '..' in Path(relative).parts or '\\' in relative:
            raise ValueError('Unsafe relative image path in manifest')
    first=frame.relative_path.iloc[0]; root=Path(options['input_root'])
    if options['dataset_root']:
        candidates=[Path(options['dataset_root'])]
    else:
        candidates=[root/'tiny-genimage',root/'datasets/yangsangtai/tiny-genimage']
        generator=Path(first).parts[0]
        for depth in range(5):
            candidates.extend(p.parent for p in root.glob('/'.join(['*']*depth+[generator])) if p.is_dir())
    matches=sorted({p.resolve() for p in candidates if (p/first).is_file()})
    if len(matches)!=1:
        raise ValueError(f'Set dataset_root explicitly. Candidates: {[str(p) for p in matches]}')
    base=matches[0]; resolved=frame.copy(); resolved['path']=[str(base/p) for p in frame.relative_path]
    for row in resolved.itertuples():
        p=Path(row.path)
        if not p.is_file() or (options['verify_file_sizes'] and p.stat().st_size!=int(row.file_bytes)):
            raise ValueError(f'Missing or changed input file: {p}')
    print(f'[manifest] {len(resolved):,} files validated by path/size, no RGB/hash scan: {base}',flush=True)
    return resolved,base


def save_csv(path,frame):
    path=Path(path); temp=path.with_suffix('.tmp')
    frame.to_csv(temp,index=False); temp.replace(path)


def bind_metadata(path,metadata):
    metadata=json.loads(json.dumps(metadata,default=str))
    if path.exists():
        previous=json.loads(path.read_text(encoding='utf-8'))
        if previous!=metadata:
            raise ValueError(f'Resume fingerprint changed: {path}. Use a new output directory.')
    else:
        dump(path,metadata)


def load_case_parts(folder,frame,batch_size):
    parts=[pd.read_csv(p) for p in sorted(folder.glob('part_*.csv'))]
    if not parts:
        return pd.DataFrame(),0
    prior=pd.concat(parts,ignore_index=True)
    n=len(prior)
    if n>len(frame) or (n<len(frame) and n%batch_size):
        raise ValueError('Resume must begin after a complete logical batch')
    expected=frame.iloc[:n]
    if prior.sample_id.tolist()!=expected.sample_id.tolist() or prior.label.tolist()!=expected.label.tolist() or prior.generator.tolist()!=expected.generator.tolist():
        raise ValueError('Saved progress does not match the ordered manifest')
    if prior.batch_id.tolist()!=[i//batch_size for i in range(n)] or not np.isfinite(prior.prob_fake).all():
        raise ValueError('Corrupt probability/batch records in progress')
    return prior,n


@torch.no_grad()
def evaluate_case(model,anchor,frame,cfg,options,name,out,fingerprint,device):
    spec=case_spec(name); folder=out/'progress'/name; folder.mkdir(parents=True,exist_ok=True)
    bind_metadata(folder/'metadata.json',dict(run=fingerprint,case=spec))
    prior,start=load_case_parts(folder,frame,cfg['test_batch'])
    if start==len(frame):
        print(f'[{name}] using completed progress',flush=True)
        return prior
    model.eval(); dataset=VariantImages(frame,cfg,spec)
    data=DataLoader(Subset(dataset,range(start,len(frame))),batch_size=cfg['test_batch'],shuffle=False,
        num_workers=options['num_workers'],collate_fn=native_collate,pin_memory=device.type=='cuda')
    pending=[]; parts=[prior] if len(prior) else []; batch0=start//cfg['test_batch']; tick=time.monotonic()
    try:
        for local_bid,(images,labels,indices) in enumerate(data):
            bid=batch0+local_bid
            bases=estimate_bases(anchor,images,cfg,device,model.selected)
            probabilities=np.empty(len(images))
            for offset,x in image_chunks(images,cfg['micro_batch']):
                with amp(cfg,device):
                    logits=model(x.to(device),bases)
                probabilities[offset:offset+len(x)]=logits.float().softmax(-1)[:,1].cpu().numpy()
            if not np.isfinite(probabilities).all():
                raise ValueError(f'Nonfinite predictions: {name}, batch {bid}')
            item=frame.iloc[indices.tolist()][['sample_id','relative_path','generator','label','pixel_hash','width','height']].copy()
            item=item.assign(case=name,model='clip_gsd',seed=cfg['seeds'][0],prob_fake=probabilities,batch_id=bid,
                input_tokens=[(x.shape[-2]//14)*(x.shape[-1]//14)+1 for x in images])
            pending.append(item)
            done=int(indices[-1])+1
            if local_bid==0 or (bid+1)%options['log_every_batches']==0:
                print(f'[{name}] batch {bid+1}/{math.ceil(len(frame)/cfg["test_batch"])}; {done:,} images',flush=True)
            if len(pending)>=options['save_every_batches'] or done==len(frame):
                chunk=pd.concat(pending,ignore_index=True)
                save_csv(folder/f'part_{int(indices[0])-sum(len(p) for p in pending[:-1]):07d}.csv',chunk)
                parts.append(chunk); pending=[]
                print(f'[{name}] {done:,}/{len(frame):,} images; batch {bid+1}; {time.monotonic()-tick:.1f}s this invocation',flush=True)
                dump(folder/'status.json',dict(completed_images=done,total=len(frame),last_batch=bid))
    except torch.cuda.OutOfMemoryError as exc:
        raise RuntimeError(f'CUDA OOM in {name}; saved progress can be resumed, unsaved batches will rerun. No automatic crop/batch/resize changes.') from exc
    result=pd.concat(parts,ignore_index=True)
    if result.sample_id.tolist()!=frame.sample_id.tolist():
        raise ValueError('Incomplete/incorrect case order')
    save_csv(out/f'predictions_{name}.csv',result)
    return result


def eval_metrics(group,threshold=.5):
    y=group.label.to_numpy(); p=group.prob_fake.to_numpy(); z=p>=threshold
    tn=int(((y==0)&~z).sum()); fp=int(((y==0)&z).sum()); fn=int(((y==1)&~z).sum()); tp=int(((y==1)&z).sum())
    real=tn/(tn+fp) if tn+fp else float('nan'); fake=tp/(tp+fn) if tp+fn else float('nan')
    both=len(np.unique(y))==2
    return dict(n=len(y),accuracy=float((y==z).mean()),balanced_accuracy=(real+fake)/2,
        real_recall=real,fake_recall=fake,fake_precision=tp/(tp+fp) if tp+fp else 0.,
        fake_f1=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0.,
        auc=float(roc_auc_score(y,p)) if both else float('nan'),
        ap=float(average_precision_score(y,p)) if both else float('nan'),tn=tn,fp=fp,fn=fn,tp=tp)


def write_eval_reports(all_predictions,cfg,out):
    rows=[]
    for case,f in all_predictions.groupby('case',sort=False):
        for gen,g in [('pooled',f),*f.groupby('generator')]:
            rows.append(dict(case=case,generator=gen,**eval_metrics(g,cfg['threshold'])))
    table=pd.DataFrame(rows); save_csv(out/'metrics.csv',table)
    per=table[table.generator!='pooled']
    per.groupby('case')[['accuracy','balanced_accuracy','real_recall','fake_recall','auc','ap']].mean().to_csv(out/'macro_all.csv')
    base=all_predictions[all_predictions.case=='native'].set_index('sample_id'); pairs=[]
    for name,f in all_predictions[all_predictions.case!='native'].groupby('case'):
        changed=f.set_index('sample_id').loc[base.index]
        if not np.array_equal(base.batch_id,changed.batch_id):
            raise ValueError('Batch membership changed across variants')
        pair=base[['generator','label','batch_id']].copy()
        pair['native_prob']=base.prob_fake; pair['changed_prob']=changed.prob_fake
        pair['delta']=pair.changed_prob-pair.native_prob
        pair['native_correct']=(pair.native_prob>=cfg['threshold']).astype(int)==pair.label
        pair['changed_correct']=(pair.changed_prob>=cfg['threshold']).astype(int)==pair.label
        save_csv(out/f'paired_{name}.csv',pair.reset_index())
        for (gen,label),g in pair.groupby(['generator','label']):
            pairs.append(dict(case=name,generator=gen,label=label,n=len(g),mean_delta=g.delta.mean(),
                mean_abs_delta=g.delta.abs().mean(),flip_rate=(g.native_correct!=g.changed_correct).mean(),
                correct_to_wrong=int((g.native_correct&~g.changed_correct).sum()),
                wrong_to_correct=int((~g.native_correct&g.changed_correct).sum())))
    save_csv(out/'paired_summary.csv',pd.DataFrame(pairs))
    delta=table.merge(table[table.case=='native'].drop(columns='case'),on='generator',suffixes=('','_native'))
    for field in ('accuracy','balanced_accuracy','real_recall','fake_recall','auc','ap'):
        delta['delta_'+field]=delta[field]-delta[field+'_native']
    save_csv(out/'delta_from_native.csv',delta)
    return table


def compare_native_reference(native,reference_path,options,out):
    if not reference_path.is_file():
        result=dict(status='unavailable',reason='No historical native predictions')
    else:
        old=pd.read_csv(reference_path)
        if 'case' in old:
            old=old[old.case=='native']
        if 'model' in old:
            old=old[old.model=='clip_gsd']
        if 'seed' in old:
            old=old[old.seed==int(native.seed.iloc[0])]
        if old.sample_id.duplicated().any() or not set(native.sample_id)<=set(old.sample_id):
            raise ValueError('Historical native reference cohort mismatch')
        old=old.set_index('sample_id').loc[native.sample_id]
        if not np.array_equal(old.label.to_numpy(),native.label.to_numpy()) or not np.array_equal(old.batch_id.to_numpy(),native.batch_id.to_numpy()):
            raise ValueError('Historical native labels/batch membership mismatch')
        delta=np.abs(native.prob_fake.to_numpy()-old.prob_fake.to_numpy())
        flips=int(((native.prob_fake.to_numpy()>=.5)!=(old.prob_fake.to_numpy()>=.5)).sum())
        result=dict(status='match' if delta.max()<=options['native_tolerance'] and flips==0 else 'review_required',
            n=len(native),max_abs_probability_delta=float(delta.max()),mean_abs_probability_delta=float(delta.mean()),prediction_flips=flips,
            note='GPU eigensolver/precision can differ slightly; investigate material native mismatch before interpreting variants')
    dump(out/'native_reference_check.json',result); print('[native reference]',result,flush=True)
    return result


def run_evaluation(options=None):
    opts={**EVAL_DEFAULTS,**(options or {})}
    if not opts['cases'] or opts['cases'][0]!='native' or len(set(opts['cases']))!=len(opts['cases']):
        raise ValueError('Cases must be unique and start with native')
    for name in opts['cases']:
        case_spec(name)
    if min(opts[k] for k in ('micro_batch','anchor_micro_batch','save_every_batches','log_every_batches'))<1 or opts['num_workers']<0:
        raise ValueError('Positive compute/checkpoint intervals required')
    if opts['max_batches'] is not None and (not isinstance(opts['max_batches'],int) or opts['max_batches']<1):
        raise ValueError('max_batches must be None or a positive integer (smoke only)')
    device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    if device.type!='cuda':
        raise RuntimeError('Full pretrained evaluation requires GPU; unit tests use tiny CPU models')
    torch.backends.cuda.matmul.allow_tf32=False; torch.backends.cudnn.allow_tf32=False
    checkpoint_path=find_checkpoint(opts); saved_root=checkpoint_path.parents[2]
    print(f'[checkpoint] {checkpoint_path}; loading CPU weights; NEVER training',flush=True)
    payload=torch.load(checkpoint_path,map_location='cpu',weights_only=False)
    cfg=payload.get('config')
    if not cfg:
        raise ValueError('Checkpoint must include training config')
    validate_raw_config(cfg)
    if cfg['input_mode']!='center_crop_max_edge_pad14' or cfg.get('crop_max_side')!=560:
        raise ValueError('Expected All7 center-crop560 checkpoint')
    if opts['expected_training_signature'] and payload.get('signature')!=opts['expected_training_signature']:
        raise ValueError('Wrong training checkpoint signature')
    selected=json.loads((checkpoint_path.parent/'selected_checkpoint.json').read_text(encoding='utf-8'))
    if selected['epoch']!=payload['epoch']+1 or selected['signature']!=payload['signature']:
        raise ValueError('Checkpoint does not match selected epoch/signature')
    manifest_path=saved_root/f"seed{cfg['seeds'][0]}/test_manifest.csv"
    frame=pd.read_csv(manifest_path)
    if len(frame)!=7000 or set(frame.generator)!=set(GENS7) or set(frame.label)!={0,1}:
        raise ValueError('Expected complete original 7,000-image All7 manifest')
    split=json.loads((manifest_path.parent/'split_summary.json').read_text(encoding='utf-8'))
    if digest(frame.sample_id.tolist())!=split['test_hash']:
        raise ValueError('Manifest ID order/hash differs from saved training split')
    if len(cfg['seeds'])!=1:
        raise ValueError('Select a single-seed checkpoint for this evaluator')
    if opts['max_batches'] is not None:
        frame=frame.iloc[:opts['max_batches']*cfg['test_batch']].copy()
    frame,base=resolve_test_files(frame,opts)
    cfg={**cfg,'micro_batch':opts['micro_batch'],'anchor_micro_batch':opts['anchor_micro_batch']}
    if cfg['precision']=='bf16' and not torch.cuda.is_bf16_supported():
        raise ValueError('Checkpoint inference precision bf16 unsupported on this GPU')
    out=Path(opts['output_root']); out.mkdir(parents=True,exist_ok=True)
    checkpoint_sha=file_sha(checkpoint_path)
    inference_cfg={k:v for k,v in cfg.items() if k not in ('dataset_root','input_root','output_root','resume','scan_log_every','eval_cases','model_id')}
    provenance=dict(checkpoint_sha256=checkpoint_sha,training_signature=payload['signature'],selected_epoch=payload['epoch']+1,
        manifest_sha256=file_sha(manifest_path),sample_order_hash=digest(frame.sample_id.tolist()),n=len(frame),
        inference_config=inference_cfg,transform_version='original_before_crop560_v1',max_batches=opts['max_batches'],
        anchor_model=opts['pretrained_model_path'] or cfg['model_id'],
        environment=dict(torch=str(torch.__version__),transformers=transformers.__version__,
            pillow=PIL.__version__,python=platform.python_version(),gpu=torch.cuda.get_device_name(0)))
    fingerprint=digest(provenance); bind_metadata(out/'provenance.json',provenance)
    dump(out/'evaluation_config.json',opts)
    dump(out/'source_paths.json',dict(checkpoint=str(checkpoint_path),dataset_root=str(base),manifest=str(manifest_path)))
    print(f"[model] recreate pretrained frozen anchor; batch={cfg['test_batch']}, native crop max={cfg['crop_max_side']}",flush=True)
    if opts['pretrained_model_path']:
        cfg.update(model_id=opts['pretrained_model_path'],local_files_only=True)
    seed_all(cfg['seeds'][0]); model,anchor=build_models(cfg,device,True)
    model.load_state_dict(payload['model'],strict=True); del payload
    model.requires_grad_(False).eval(); anchor.requires_grad_(False).eval()
    predictions=[]; ref=None
    for name in opts['cases']:
        result=evaluate_case(model,anchor,frame,cfg,opts,name,out,fingerprint,device)
        save_csv(out/f'predictions_{name}.csv',result)
        predictions.append(result)
        write_eval_reports(pd.concat(predictions,ignore_index=True),cfg,out)
        if name=='native':
            reference=checkpoint_path.parent/'predictions_native.csv'
            if not reference.is_file():
                reference=saved_root/'predictions.csv'
            ref=compare_native_reference(result,reference,opts,out)
        gc.collect(); torch.cuda.empty_cache()
    all_predictions=pd.concat(predictions,ignore_index=True); save_csv(out/'predictions.csv',all_predictions)
    table=write_eval_reports(all_predictions,cfg,out)
    dump(out/'completion.json',dict(status='complete',training=False,smoke=opts['max_batches'] is not None,
        evaluated_cases=opts['cases'],n_per_case=len(frame),fingerprint=fingerprint,native_reference=ref))
    print(table[table.generator=='pooled'].to_string(index=False),flush=True)
    return table


"""Inference-only uploaded-checkpoint evaluator with bounded, resumable sessions."""
if 'case_spec' not in globals():
    from pathlib import Path
    _here = Path(__file__).resolve().parent
    for _name in ('runtime.py', 'runtime_raw_all7.py', 'manifest_reuse.py', 'evaluate_all7_variants.py'):
        exec(compile((_here / _name).read_text(encoding='utf-8'), str(_here / _name), 'exec'), globals())

import shutil
import zipfile

UPLOAD_EVALUATOR = 'gsd_uploaded_best_variants_v1'
UPLOADED_FULL_SUITE_CASES = (
    'native', 'fake_jpeg96', 'all_jpeg96', 'all_jpeg90', 'all_jpeg80',
    'resize_square256', 'resize_square560', 'blur_sigma1', 'resize256_jpeg96',
)
UPLOAD_DEFAULTS = dict(
    input_root='/kaggle/input', checkpoint_path=None, test_manifest_path=None, dataset_root=None,
    output_root='/kaggle/working/gsd_uploaded_best_variants_v1',
    checkpoint_cache='/kaggle/working/gsd_uploaded_checkpoint_cache',
    pretrained_model_path=None, resume_from='auto',
    expected_signature='4595045b4c6c76c1ad4aa303a7d2f1690cfac8150add4548d24609a6c980dce6',
    expected_epoch=1,
    expected_test_order_hash='c544b38bf4276a2fce06fd205fb4dbb86fa772ac26532509679311e7d8a96aa2',
    cases=['native', 'fake_jpeg96', 'all_jpeg96', 'all_jpeg90', 'all_jpeg80',
           'resize_square256', 'resize_square560', 'blur_sigma1', 'resize256_jpeg96'],
    micro_batch=1, anchor_micro_batch=1, num_workers=0,
    save_every_batches=10, log_every_batches=10,
    max_new_cases_per_run=3, time_budget_hours=9.5, max_batches=None,
    verify_file_sizes=True,
)


def finalize_uploaded_notebook_config(options=None, smoke_only=False):
    """Pin the shipped GPU notebook's full suite at the actual invocation boundary.

    A stale cell's cases=['native'], max_batches=2 or case limit cannot silently
    turn the full evaluation into a successful native-only/smoke run. The general
    run_uploaded_evaluation API still accepts custom cases and case budgets.
    """
    if not isinstance(smoke_only, bool):
        raise ValueError('SMOKE_ONLY must be explicitly True or False')
    opts = dict(UPLOAD_DEFAULTS, **(options or {}))
    if smoke_only:
        opts.update(cases=['native', 'fake_jpeg96'], max_batches=2,
                    max_new_cases_per_run=2, resume_from=None,
                    output_root='/kaggle/working/gsd_uploaded_best_variants_smoke_v2')
    else:
        opts.update(cases=list(UPLOADED_FULL_SUITE_CASES), max_batches=None,
                    max_new_cases_per_run=None)
    validate_upload_options(opts)
    print('[launch] ' + ('SMOKE ONLY: 16 images/case; NOT full test results.' if smoke_only else
          'FULL SUITE: 9 cases, 7,000 images/case; no per-session case-count limit.'), flush=True)
    print('[launch] ACTUAL CASES: ' + ', '.join(opts['cases']), flush=True)
    print(f'[launch] time_budget_hours={opts["time_budget_hours"]}; '
          f'resume_from={opts["resume_from"]}; output={opts["output_root"]}', flush=True)
    return opts


def atomic_json(path, value):
    path = Path(path)
    temp = path.with_suffix(path.suffix + '.tmp')
    temp.write_text(json.dumps(value, ensure_ascii=False, indent=2, default=str), encoding='utf-8')
    temp.replace(path)


def bind_upload_metadata(path, metadata):
    metadata = json.loads(json.dumps(metadata, default=str))
    if path.exists():
        if json.loads(path.read_text(encoding='utf-8')) != metadata:
            raise ValueError(f'Inference fingerprint changed: {path}. Use a NEW output_root.')
    else:
        atomic_json(path, metadata)


def pytorch_zip_prefix(names):
    prefixes = [name[:-len('data.pkl')] for name in names if name.endswith('/data.pkl')]
    matches = [p for p in prefixes if p + 'version' in names and any(n.startswith(p + 'data/') for n in names)]
    return matches[0] if len(matches) == 1 else None


def materialize_checkpoint(source, cache):
    """Accept .pt, a torch archive named .zip, outer ZIP, or unpacked torch internals.

    No extractall and no direct pickle execution. Only torch.load(weights_only=True)
    interprets checkpoint metadata later. The source input is never modified.
    """
    source, cache = Path(source), Path(cache)
    if source.is_file():
        if not zipfile.is_zipfile(source):
            return source  # legacy torch serialization; restricted loader still applies
        with zipfile.ZipFile(source) as archive:
            if pytorch_zip_prefix(archive.namelist()):
                return source  # best.zip can already BE a PyTorch checkpoint
            entries = [i for i in archive.infolist() if not i.is_dir() and
                       PurePosixPath(i.filename).name in ('best.pt', 'best.pth')]
            if len(entries) != 1:
                raise ValueError(f'{source}: expected torch archive or a ZIP containing exactly one best.pt')
            item = entries[0]
            if item.file_size <= 0 or item.file_size > 4 * 1024**3:
                raise ValueError('Unexpected checkpoint archive entry size')
            cache.mkdir(parents=True, exist_ok=True)
            target = cache / ('extracted_' + digest(str(source.resolve()))[:12] + '.pt')
            temp = target.with_suffix('.tmp')
            with archive.open(item) as inp, temp.open('wb') as dest:
                shutil.copyfileobj(inp, dest, length=8 * 1024 * 1024)
            temp.replace(target)
            return target
    if source.is_dir() and (source / 'data.pkl').is_file() and (source / 'version').is_file():
        # Kaggle may unzip best.zip on upload. Restore the torch container, not a model.
        files = sorted(p for p in source.rglob('*') if p.is_file())
        if not (source / 'data').is_dir():
            raise ValueError('Unpacked checkpoint has no tensor-data directory')
        cache.mkdir(parents=True, exist_ok=True)
        target = cache / ('repacked_' + digest(str(source.resolve()))[:12] + '.pt')
        temp = target.with_suffix('.tmp')
        with zipfile.ZipFile(temp, 'w', compression=zipfile.ZIP_STORED) as archive:
            for path in files:
                relative = path.relative_to(source).as_posix()
                if path.is_symlink() or not (relative in {
                        'data.pkl', 'version', 'byteorder', '.format_version', '.storage_alignment',
                        '.data/serialization_id'} or re.fullmatch(r'data/\d+', relative)):
                    raise ValueError(f'Unexpected unpacked checkpoint file: {relative}')
                info = zipfile.ZipInfo('best/' + relative, date_time=(1980, 1, 1, 0, 0, 0))
                with path.open('rb') as inp, archive.open(info, 'w', force_zip64=True) as dest:
                    shutil.copyfileobj(inp, dest, length=8 * 1024 * 1024)
        temp.replace(target)
        return target
    raise FileNotFoundError(f'No checkpoint at {source}. Select best.pt, best.zip, or unpacked best/ folder.')


def restricted_checkpoint_load(path):
    # File extension is irrelevant. Do not fall back to unrestricted pickle loading.
    return torch.load(path, map_location='cpu', weights_only=True, mmap=zipfile.is_zipfile(path))


def validate_uploaded_payload(payload, options):
    if not isinstance(payload, dict) or not {'model', 'config', 'epoch', 'signature'} <= set(payload):
        raise ValueError('Checkpoint must contain model/config/epoch/signature')
    cfg = payload['config']
    validate_raw_config(dict(cfg, jpeg_probability=0., blur_probability=0., horizontal_flip_probability=0.))
    if cfg['input_mode'] != 'center_crop_max_edge_pad14' or cfg.get('crop_max_side') != 560:
        raise ValueError('Expected All7 crop560 checkpoint')
    if cfg.get('smoke') or len(cfg['seeds']) != 1 or cfg['test_batch'] != 8:
        raise ValueError('Expected full single-seed checkpoint with logical test batch8')
    if options['expected_signature'] and payload['signature'] != options['expected_signature']:
        raise ValueError(f'Wrong checkpoint signature: {payload["signature"]}')
    if options['expected_epoch'] is not None and payload['epoch'] + 1 != options['expected_epoch']:
        raise ValueError('Unexpected checkpoint epoch')
    if not isinstance(payload['model'], dict) or not payload['model']:
        raise ValueError('Missing model state_dict')
    return dict(cfg)


def find_uploaded_checkpoint(options):
    if options['checkpoint_path']:
        candidates = [Path(options['checkpoint_path'])]
    else:
        root = Path(options['input_root'])
        candidates = sorted({p for pattern in ('best.pt', 'best.pth', 'best.zip') for p in root.rglob(pattern)
                             if p.is_file()} | {p.parent for p in root.rglob('data.pkl')
                             if (p.parent / 'version').is_file() and (p.parent / 'data').is_dir()})
    matches, failures = [], []
    for source in candidates:
        try:
            path = materialize_checkpoint(source, options['checkpoint_cache'])
            payload = restricted_checkpoint_load(path)
            validate_uploaded_payload(payload, options)
            matches.append((source, path, payload))
        except Exception as exc:
            failures.append(f'{source}: {type(exc).__name__}: {exc}')
    if not matches:
        raise ValueError('No matching uploaded checkpoint. Upload best.pt (preferred; do not unzip it), '
                         'or set checkpoint_path explicitly. Inspected: ' + '; '.join(failures))
    if len(matches) > 1:
        hashes = {file_sha(path) for _, path, _ in matches}
        if len(hashes) != 1:
            raise ValueError('Multiple matching but different checkpoints; set checkpoint_path explicitly')
        print('[checkpoint] Identical checkpoint copies found; using the first.', flush=True)
    return matches[0]


def choose_test_manifest(cfg, options):
    explicit = options['test_manifest_path']
    if explicit:
        paths = [Path(explicit)]
    else:
        embedded = Path(cfg.get('manifest_root') or '/nonexistent') / f'seed{cfg["seeds"][0]}' / 'test_manifest.csv'
        paths = [embedded] if embedded.is_file() else sorted(Path(options['input_root']).rglob('test_manifest.csv'))
    valid, failures = [], []
    for path in paths:
        try:
            frame = pd.read_csv(path, dtype={k: str for k in
                ('sample_id', 'relative_path', 'generator', 'split', 'file_hash', 'pixel_hash')})
            frame = validate_manifest_frame(frame, cfg, 'test')
            if len(frame) != 7000:
                raise ValueError('Expected 7,000 test images')
            order = digest(frame.sample_id.tolist())
            if options['expected_test_order_hash'] and order != options['expected_test_order_hash']:
                raise ValueError('Test ID order does not match the saved All7 cohort')
            summary_path = path.parent / 'split_summary.json'
            if summary_path.is_file():
                summary = json.loads(summary_path.read_text(encoding='utf-8'))
                if summary.get('test') != len(frame) or summary.get('test_hash') != order:
                    raise ValueError('Saved test split_summary does not match')
            fields = sorted(MANIFEST_COLUMNS)
            valid.append((path, frame, digest(frame[fields].to_dict('records'))))
        except Exception as exc:
            failures.append(f'{path}: {exc}')
    if not valid or len({identity for _, _, identity in valid}) != 1:
        raise ValueError('Set test_manifest_path to the matching seed42/test_manifest.csv. '
                         'Missing/different cohorts: ' + '; '.join(failures))
    if len(valid) > 1:
        print('[manifest] Multiple identical test manifests; reusing one, no image scan.', flush=True)
    return valid[0]


def canonical_case(name):
    spec = case_spec(name)
    if spec['kind'] in ('native', 'png'):
        return 'native'
    if spec['kind'] in ('all_jpeg', 'fake_jpeg'):
        return spec['kind'] + str(spec['quality'])
    if spec['kind'] == 'resize_square':
        return 'resize_square' + str(spec['side'])
    return name


def plan_uploaded_cases(requested):
    if not requested:
        raise ValueError('Select at least one case')
    cases, aliases, duplicates = ['native'], {}, []
    seen_requests = set()
    for name in requested:
        canonical = canonical_case(name)
        if name in seen_requests:
            duplicates.append(name)
        seen_requests.add(name)
        if canonical != name:
            aliases[name] = canonical
        if canonical not in cases:
            cases.append(canonical)
    return dict(requested=requested, cases=cases, aliases_not_measured=aliases, duplicate_requests=duplicates,
                note='Native always included. PNG roundtrip is an RGB-pixel identity alias, '
                     'not independently measured. Fake-only JPEG is NOT equivalent to all-image JPEG.')


def validate_upload_options(options):
    for key in ('micro_batch', 'anchor_micro_batch', 'save_every_batches', 'log_every_batches'):
        if isinstance(options[key], bool) or not isinstance(options[key], int) or options[key] < 1:
            raise ValueError(f'{key} must be a positive integer')
    for key in ('max_new_cases_per_run', 'max_batches'):
        if options[key] is not None and (isinstance(options[key], bool) or not isinstance(options[key], int) or options[key] < 1):
            raise ValueError(f'{key} must be None or a positive integer')
    if options['num_workers'] < 0 or not 0 < options['time_budget_hours'] <= 10.5:
        raise ValueError('Nonnegative workers and time_budget_hours in (0,10.5] required for a 12h session')
    return plan_uploaded_cases(options['cases'])


def restore_uploaded_progress(out, options, fingerprint):
    if any((out / 'progress').glob('*/part_*.csv')):
        print('[resume] Existing local progress takes precedence.', flush=True)
        return None
    requested = options['resume_from']
    if not requested:
        return None
    if requested == 'auto':
        states = []
        for path in Path(options['input_root']).rglob('evaluation_state.json'):
            try:
                state = json.loads(path.read_text(encoding='utf-8'))
                if state.get('evaluator') == UPLOAD_EVALUATOR and state.get('fingerprint') == fingerprint:
                    states.append(path)
            except (ValueError, OSError):
                continue
        if not states:
            return None
        # Prefer the one with the most persisted rows, then use deterministic path ordering.
        states.sort(key=lambda p: (sum(json.loads(p.read_text(encoding='utf-8')).get('saved_rows', {}).values()), str(p)))
        source = states[-1].parent
    else:
        source = Path(requested)
    state = json.loads((source / 'evaluation_state.json').read_text(encoding='utf-8'))
    if state.get('evaluator') != UPLOAD_EVALUATOR or state.get('fingerprint') != fingerprint:
        raise ValueError('Resume source has a different checkpoint/cohort/processing/environment fingerprint')
    for path in sorted((source / 'progress').rglob('*')):
        if not path.is_file() or path.suffix not in ('.csv', '.json'):
            continue
        relative = path.relative_to(source)
        if path.is_symlink():
            raise ValueError('Refusing a symlink in saved progress')
        target = out / relative
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(path, target)
    print(f'[resume] Restored saved complete-batch progress from {source}', flush=True)
    return source


def read_uploaded_case(folder, frame, cfg, name):
    if any(folder.glob('part_*.csv')) and not (folder / 'metadata.json').is_file():
        raise ValueError('Saved progress has no case fingerprint metadata; refusing unverified cache')
    prior, n = load_case_parts(folder, frame, cfg['test_batch'])
    if n:
        required = {'logit_real', 'logit_fake', 'logit_margin', 'input_tokens', 'case', 'seed', 'model'}
        if not required <= set(prior) or not np.isfinite(prior[list(required - {'case', 'model'})].to_numpy()).all():
            raise ValueError('Saved progress lacks finite raw logits/token metadata')
        if set(prior.case) != {name} or set(prior.seed) != {cfg['seeds'][0]} or set(prior.model) != {'clip_gsd'}:
            raise ValueError('Saved case/seed/model mismatch')
        if not np.allclose(prior.logit_fake - prior.logit_real, prior.logit_margin, rtol=1e-5, atol=1e-5):
            raise ValueError('Saved raw-logit margin mismatch')
        expected_scores = torch.from_numpy(prior[['logit_real', 'logit_fake']].to_numpy()).softmax(-1)[:, 1].numpy()
        if not np.allclose(prior.prob_fake, expected_scores, rtol=1e-6, atol=2e-7):
            raise ValueError('Saved scores do not match raw logits')
    return prior, n


@torch.no_grad()
def evaluate_uploaded_case(model, anchor, frame, cfg, options, name, out, fingerprint,
                           device, deadline, callback=None, backend=None):
    folder = out / 'progress' / name
    folder.mkdir(parents=True, exist_ok=True)
    if any(folder.glob('part_*.csv')) and not (folder / 'metadata.json').is_file():
        raise ValueError('Saved progress has no case fingerprint metadata; refusing unverified cache')
    bind_upload_metadata(folder / 'metadata.json', dict(run=fingerprint, case=case_spec(name), evaluator=UPLOAD_EVALUATOR))
    prior, start = read_uploaded_case(folder, frame, cfg, name)
    if start == len(frame):
        return prior, True
    model.eval()
    data = DataLoader(Subset(VariantImages(frame, cfg, case_spec(name)), range(start, len(frame))),
                      batch_size=cfg['test_batch'], shuffle=False, num_workers=options['num_workers'],
                      collate_fn=native_collate, pin_memory=device.type == 'cuda', drop_last=False)
    pending, parts, first_pending, done = [], [prior] if start else [], start, start

    def flush():
        nonlocal pending
        if pending:
            chunk = pd.concat(pending, ignore_index=True)
            save_csv(folder / f'part_{first_pending:07d}.csv', chunk)
            parts.append(chunk)
            pending = []
            if callback:
                callback(name, done)

    try:
        for local_bid, (images, labels, indices) in enumerate(data):
            if time.monotonic() >= deadline:
                break
            bid = start // cfg['test_batch'] + local_bid
            if backend is not None:
                scores = backend.predict_batch(model, anchor, images, cfg, device)
                if scores.shape != (len(images), 2) or not np.isfinite(scores).all():
                    raise FloatingPointError(f'Invalid backend logits: {name}, batch {bid}')
                # Preserve softmax-of-FP32-logits semantics; no TPU host round-trip per scalar.
                probabilities = torch.from_numpy(scores).float().softmax(-1)[:, 1].numpy()
            else:
                bases = estimate_bases(anchor, images, cfg, device, model.selected)
                scores = np.empty((len(images), 2), dtype=np.float32)
                probabilities = np.empty(len(images), dtype=np.float32)
                for offset, x in image_chunks(images, cfg['micro_batch']):
                    with amp(cfg, device):
                        logits = model(x.to(device), bases)
                    if not torch.isfinite(logits).all():
                        raise FloatingPointError(f'Nonfinite logits: {name}, batch {bid}')
                    scores[offset:offset + len(x)] = logits.float().cpu().numpy()
                    probabilities[offset:offset + len(x)] = logits.float().softmax(-1)[:, 1].cpu().numpy()
            item = frame.iloc[indices.tolist()][['sample_id', 'relative_path', 'generator', 'label',
                                                'pixel_hash', 'width', 'height']].copy()
            item = item.assign(case=name, model='clip_gsd', seed=cfg['seeds'][0], prob_fake=probabilities,
                               logit_real=scores[:, 0], logit_fake=scores[:, 1],
                               logit_margin=scores[:, 1] - scores[:, 0], batch_id=bid,
                               input_tokens=[(x.shape[-2] // 14) * (x.shape[-1] // 14) + 1 for x in images])
            if not pending:
                first_pending = int(indices[0])
            pending.append(item)
            done = int(indices[-1]) + 1
            if local_bid == 0 or (bid + 1) % options['log_every_batches'] == 0:
                print(f'[{name}] {done:,}/{len(frame):,} images; batch {bid + 1}/{math.ceil(len(frame)/cfg["test_batch"])}', flush=True)
            if len(pending) >= options['save_every_batches'] or done == len(frame):
                flush()
    finally:
        # Soft interrupts / planned stop retain completed batches. Hard kills cannot run finally.
        flush()
    result = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame()
    complete = len(result) == len(frame)
    if len(result):
        expected = frame.iloc[:len(result)]
        if result.sample_id.tolist() != expected.sample_id.tolist():
            raise ValueError('Incorrect resumed case order')
        save_csv(out / f'predictions_{name}{"" if complete else ".partial"}.csv', result)
    return result, complete


def run_uploaded_evaluation(options=None, backend=None):
    started = time.monotonic()
    opts = dict(UPLOAD_DEFAULTS, **(options or {}))
    plan = validate_upload_options(opts)
    deadline = started + opts['time_budget_hours'] * 3600
    source, checkpoint_path, payload = find_uploaded_checkpoint(opts)
    cfg = validate_uploaded_payload(payload, opts)
    manifest_path, full_frame, manifest_identity = choose_test_manifest(cfg, opts)
    frame = full_frame.iloc[:opts['max_batches'] * cfg['test_batch']].copy() if opts['max_batches'] else full_frame.copy()
    frame, dataset_root = resolve_test_files(frame, opts)
    cfg.update(micro_batch=opts['micro_batch'], anchor_micro_batch=opts['anchor_micro_batch'])
    if backend is not None:
        device = backend.prepare(cfg, opts)
        environment = backend.environment()
    else:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        if device.type != 'cuda':
            raise RuntimeError('Use Kaggle GPU; offline CPU tests use tiny models only')
        if cfg['precision'] == 'bf16' and not torch.cuda.is_bf16_supported():
            raise ValueError('Checkpoint precision bf16 unsupported on this GPU')
        torch.backends.cuda.matmul.allow_tf32 = False
        torch.backends.cudnn.allow_tf32 = False
        environment = dict(torch=str(torch.__version__), transformers=transformers.__version__,
                           pillow=PIL.__version__, gpu=torch.cuda.get_device_name(0))
    out = Path(opts['output_root'])
    out.mkdir(parents=True, exist_ok=True)
    checkpoint_sha = file_sha(checkpoint_path)
    critical = {k: cfg[k] for k in ('input_mode', 'crop_max_side', 'test_batch', 'micro_batch',
                'anchor_micro_batch', 'precision', 'basis_method', 'attention', 'pooling', 'k',
                'lambda_max', 'a_init', 'b_init', 'jpeg_subsampling', 'threshold')}
    identity = dict(evaluator=UPLOAD_EVALUATOR, checkpoint_sha256=checkpoint_sha,
        training_signature=payload['signature'], epoch=payload['epoch'] + 1,
        manifest_identity=manifest_identity, sample_order_hash=digest(frame.sample_id.tolist()), n=len(frame),
        inference_config=critical, transform_version='original_rgb_before_crop560_v1',
        pretrained_anchor=opts['pretrained_model_path'] or cfg['model_id'],
        environment=environment)
    fingerprint = digest(identity)
    bind_upload_metadata(out / 'run_fingerprint.json', identity)
    restored = restore_uploaded_progress(out, opts, fingerprint)
    atomic_json(out / 'case_plan.json', plan)
    atomic_json(out / 'evaluation_config.json', opts)
    atomic_json(out / 'checkpoint_inspection.json', dict(
        source=str(source), materialized_checkpoint=str(checkpoint_path), sha256=checkpoint_sha,
        epoch=payload['epoch'] + 1, signature=payload['signature'], protocol=payload.get('protocol'),
        training_config=payload['config'], validation=payload.get('validation'),
        val_loss=payload.get('val_loss'), tensors=len(payload['model']),
        note='Best saved checkpoint among completed validations, NOT proof the full training run finished.'))
    atomic_json(out / 'source_paths.json', dict(manifest=str(manifest_path), dataset_root=str(dataset_root),
                                              checkpoint=str(source), restored_from=str(restored) if restored else None))
    state = dict(evaluator=UPLOAD_EVALUATOR, fingerprint=fingerprint, status='running',
                 planned_cases=plan['cases'], completed_cases=[], saved_rows={}, n_per_case=len(frame),
                 smoke=opts['max_batches'] is not None, training=False)

    def progress(name, done):
        state['saved_rows'][name] = done
        atomic_json(out / 'evaluation_state.json', state)

    atomic_json(out / 'evaluation_state.json', state)
    print(f'[checkpoint] epoch {payload["epoch"] + 1}, signature={payload["signature"]}; INFERENCE ONLY', flush=True)
    print(f'[cases] {plan["cases"]}; aliases skipped={plan["aliases_not_measured"]}', flush=True)
    print(f'[budget] Max {opts["max_new_cases_per_run"]} new cases / {opts["time_budget_hours"]}h; '
          'planned pauses return normally so Kaggle can save outputs.', flush=True)
    complete_predictions, model, anchor, started_cases, reason = [], None, None, 0, 'complete'
    for name in plan['cases']:
        folder = out / 'progress' / name
        prior, n = read_uploaded_case(folder, frame, cfg, name)
        if n:
            bind_upload_metadata(folder / 'metadata.json', dict(run=fingerprint, case=case_spec(name), evaluator=UPLOAD_EVALUATOR))
        if n == len(frame):
            print(f'[{name}] reuse COMPLETE saved case; zero model forward.', flush=True)
            result, complete = prior, True
        else:
            if time.monotonic() >= deadline:
                reason = 'time_budget'
                break
            if opts['max_new_cases_per_run'] is not None and started_cases >= opts['max_new_cases_per_run']:
                reason = 'case_limit'
                break
            if model is None:
                if opts['pretrained_model_path']:
                    cfg.update(model_id=opts['pretrained_model_path'], local_files_only=True)
                seed_all(cfg['seeds'][0])
                if backend is None:
                    model, anchor = build_models(cfg, device, True)
                    model.load_state_dict(payload['model'], strict=True)
                else:
                    model, anchor = backend.build_models(cfg, device, payload['model'])
                del payload
                model.requires_grad_(False).eval()
                anchor.requires_grad_(False).eval()
            started_cases += 1
            arguments = (model, anchor, frame, cfg, opts, name, out, fingerprint, device, deadline, progress)
            result, complete = (evaluate_uploaded_case(*arguments) if backend is None else
                                evaluate_uploaded_case(*arguments, backend=backend))
        progress(name, len(result))
        if not complete:
            reason = 'time_budget'
            if len(result):
                partial_rows = [dict(case=name, generator=gen, complete=False, **eval_metrics(group, cfg['threshold']))
                                for gen, group in [('pooled', result), *result.groupby('generator')]]
                save_csv(out / 'partial_metrics.csv', pd.DataFrame(partial_rows))
            break
        save_csv(out / f'predictions_{name}.csv', result)
        complete_predictions.append(result)
        state['completed_cases'].append(name)
        table = write_eval_reports(pd.concat(complete_predictions, ignore_index=True), cfg, out)
        print(table[(table.case == name) & (table.generator == 'pooled')].to_string(index=False), flush=True)
        gc.collect()
        if backend is None:
            torch.cuda.empty_cache()
        else:
            backend.cleanup()
    all_predictions = pd.concat(complete_predictions, ignore_index=True) if complete_predictions else pd.DataFrame()
    if len(all_predictions):
        save_csv(out / 'predictions.csv', all_predictions)
        table = write_eval_reports(all_predictions, cfg, out)
    else:
        table = pd.DataFrame(columns=['case', 'generator', 'n', 'accuracy', 'balanced_accuracy', 'auc', 'ap'])
    finished = len(state['completed_cases']) == len(plan['cases'])
    state.update(status='complete' if finished else 'paused', reason='complete' if finished else reason,
                 remaining_cases=[c for c in plan['cases'] if c not in state['completed_cases']],
                 elapsed_seconds=time.monotonic() - started)
    atomic_json(out / 'evaluation_state.json', state)
    atomic_json(out / 'completion.json', state)
    print(f'[session] {state["status"]}; completed={state["completed_cases"]}; remaining={state["remaining_cases"]}', flush=True)
    if not finished:
        print('[next] Save this notebook output; attach it as Input next run. resume_from=auto restores '
              'matching progress. DO NOT restart the same 3 cases without attaching saved progress.', flush=True)
    return table

## Nhận checkpoint train trên B-Free

In [ ]:
# === Nhận checkpoint GSD train trên B-Free (gsd_bfree_single_branch_paired_v1) ===============
# Evaluator bên trên là bản GỐC đã dùng cho GSD-aug-Tiny. Cell này chỉ đổi bước kiểm tra checkpoint:
# đọc config['eval_compat_config'] (do notebook train ghi), kiểm tra mọi khóa inference trùng với
# config train, rồi gọi lại đúng validate_uploaded_payload gốc. Không đổi preprocessing, batch, metric.
# train_generators trong compat config chỉ mô tả layout manifest TEST Tiny, không phải dữ liệu train.
BFREE_TRAINED_PROTOCOL = 'gsd_bfree_single_branch_paired_v1'
BFREE_INFERENCE_KEYS = ('model_id', 'input_mode', 'crop_max_side', 'test_batch', 'micro_batch',
                        'anchor_micro_batch', 'precision', 'basis_method', 'attention', 'pooling', 'k',
                        'lambda_max', 'a_init', 'b_init', 'jpeg_subsampling', 'threshold')
if '_tiny_validate_uploaded_payload' not in globals():
    _tiny_validate_uploaded_payload = validate_uploaded_payload


def validate_uploaded_payload(payload, options):
    if not isinstance(payload, dict) or not isinstance(payload.get('config'), dict):
        raise ValueError('Checkpoint must contain model/config/epoch/signature')
    cfg = payload['config']
    if cfg.get('training_protocol') != BFREE_TRAINED_PROTOCOL:
        raise ValueError('Notebook này chỉ chấm checkpoint GSD train trên B-Free (best.pt / checkpoint_veN.pt)')
    compat = cfg.get('eval_compat_config')
    if not isinstance(compat, dict):
        raise ValueError('Checkpoint thiếu eval_compat_config')
    for key in BFREE_INFERENCE_KEYS:
        if compat.get(key) != cfg.get(key):
            raise ValueError(f'eval_compat_config khác config train ở khóa {key}')
    print(f"[compat] B-Free-trained checkpoint: branch={cfg['train_branch']}, "
          f"VE{payload['epoch'] + 1}, images_seen={payload.get('images_seen')}", flush=True)
    return _tiny_validate_uploaded_payload(dict(payload, config=compat), options)

## Tìm input (giới hạn độ sâu) và resume

In [ ]:
# === Tìm input có giới hạn độ sâu (không quét ~430k ảnh B-Free bằng rglob) + resume Tiny có giới hạn ===
import os, hashlib
COMBINED_T0 = time.time()
SKIP_DIR_NAMES = {'COCO_real_512', 'SD2.1_selfconditioned', 'SD2.1_selfconditioned_origBG',
                  'SD2.1_inpainted_samecat', 'SD2.1_inpainted_samecat_origBG', 'SD2.1_inpainted_diffcat',
                  'SD2.1_inpainted_diffcat_origBG', 'progress', 'data', '.data', 'ai', 'nature'}


def bounded_dirs(root='/kaggle/input', max_depth=6):
    pending = [(Path(root), 0)]
    while pending:
        folder, depth = pending.pop()
        yield folder
        if depth >= max_depth:
            continue
        try:
            children = sorted(p for p in folder.iterdir() if p.is_dir() and not p.is_symlink())
        except OSError:
            continue
        pending.extend((p, depth + 1) for p in reversed(children) if p.name not in SKIP_DIR_NAMES)


def file_digest(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for block in iter(lambda: f.read(8 << 20), b''):
            h.update(block)
    return h.hexdigest()


def locate_checkpoint(name, input_root='/kaggle/input'):
    hits = [d / name for d in bounded_dirs(input_root) if (d / name).is_file() and (d / 'run_state.json').is_file()]
    if not hits:
        raise FileNotFoundError(f'Không thấy {name} cạnh run_state.json trong /kaggle/input. Add dataset checkpoint '
                                'hoặc đặt CHECKPOINT_PATH thủ công.')
    if len({p.stat().st_size for p in hits}) != 1:
        raise ValueError(f'Nhiều {name} khác nhau: {hits}. Đặt CHECKPOINT_PATH thủ công.')
    return hits[0]


def locate_test_manifest(input_root='/kaggle/input'):
    hits = [d / 'test_manifest.csv' for d in bounded_dirs(input_root)
            if d.name == 'seed42' and (d / 'test_manifest.csv').is_file()]
    if not hits:
        raise FileNotFoundError('Không thấy seed42/test_manifest.csv: Add dataset hadinho0710/mainfest-train-genimage.')
    if len({file_digest(p) for p in hits}) != 1:
        raise ValueError(f'Nhiều test_manifest.csv khác nhau: {hits}. Đặt TINY_TEST_MANIFEST thủ công.')
    return hits[0]


def locate_tiny_root(manifest_path, input_root='/kaggle/input'):
    first = pd.read_csv(manifest_path, nrows=1, dtype=str).relative_path.iloc[0]
    hits = sorted({d.resolve() for d in bounded_dirs(input_root, max_depth=5) if (d / first).is_file()})
    if len(hits) != 1:
        raise ValueError(f'Cần đúng một thư mục Tiny-GenImage chứa {first}; tìm thấy {hits}. Đặt TINY_ROOT thủ công.')
    return hits[0]


if '_original_restore_uploaded_progress' not in globals():
    _original_restore_uploaded_progress = restore_uploaded_progress


def restore_uploaded_progress(out, options, fingerprint):
    """Same semantics as the original 'auto' resume, but searches prior outputs with a bounded walk."""
    if options.get('resume_from') != 'auto' or any((out / 'progress').glob('*/part_*.csv')):
        return _original_restore_uploaded_progress(out, options, fingerprint)
    found = []
    for d in bounded_dirs(options['input_root']):
        path = d / 'evaluation_state.json'
        if path.is_file():
            try:
                state = json.loads(path.read_text(encoding='utf-8'))
            except (ValueError, OSError):
                continue
            if state.get('evaluator') == UPLOAD_EVALUATOR and state.get('fingerprint') == fingerprint:
                found.append((sum(state.get('saved_rows', {}).values()), str(d), d))
    if not found:
        print('[resume] Tiny: không có tiến độ cũ khớp fingerprint; chạy từ đầu.', flush=True)
        return None
    return _original_restore_uploaded_progress(out, dict(options, resume_from=str(sorted(found)[-1][2])), fingerprint)

## Cấu hình

In [ ]:
SMOKE_ONLY = False              # True = thử nhanh 16 ảnh/case, output smoke riêng (KHÔNG phải kết quả)
CHECKPOINT_NAME = 'best.pt'     # 'best.pt' (VE2) hoặc 'checkpoint_ve1.pt' / 'checkpoint_ve2.pt'
TOTAL_BUDGET_HOURS = 11.3

# None = tự tìm (giới hạn độ sâu). Điền tay nếu báo lỗi:
CHECKPOINT_PATH = None          # '/kaggle/input/datasets/hoangha071005/check-point-gsd-train-bfee-samecat-ori/gsd_bfree_samecat_origbg_v1/best.pt'
TINY_TEST_MANIFEST = None       # '/kaggle/input/datasets/hadinho0710/mainfest-train-genimage/seed42/test_manifest.csv'
TINY_ROOT = None                # '/kaggle/input/datasets/yangsangtai/tiny-genimage'

CHECKPOINT_PATH = Path(CHECKPOINT_PATH) if CHECKPOINT_PATH else locate_checkpoint(CHECKPOINT_NAME)
TAG = Path(CHECKPOINT_NAME).stem.replace('checkpoint_', '')
TINY_OUT = f'/kaggle/working/gsd_bfree_samecat_origbg_{TAG}_tiny9_v1'
TINY_TEST_MANIFEST = Path(TINY_TEST_MANIFEST) if TINY_TEST_MANIFEST else locate_test_manifest()
TINY_ROOT = Path(TINY_ROOT) if TINY_ROOT else locate_tiny_root(TINY_TEST_MANIFEST)
print('checkpoint :', CHECKPOINT_PATH)
print('manifest   :', TINY_TEST_MANIFEST)
print('tiny root  :', TINY_ROOT)
print('output     :', TINY_OUT)

## Chạy Tiny 9 case

In [ ]:
budget = min(10.5, TOTAL_BUDGET_HOURS - (time.time() - COMBINED_T0) / 3600 - 0.15)
EVAL_CONFIG = dict(UPLOAD_DEFAULTS)
EVAL_CONFIG.update(checkpoint_path=str(CHECKPOINT_PATH), test_manifest_path=str(TINY_TEST_MANIFEST),
                   dataset_root=str(TINY_ROOT), output_root=TINY_OUT, resume_from='auto',
                   expected_signature=None, expected_epoch=None, time_budget_hours=round(budget, 3),
                   micro_batch=1, anchor_micro_batch=1, num_workers=0)
EVAL_CONFIG = finalize_uploaded_notebook_config(EVAL_CONFIG, smoke_only=SMOKE_ONLY)
print(f'[Tiny] budget {EVAL_CONFIG["time_budget_hours"]:.2f} h', flush=True)
metrics = run_uploaded_evaluation(EVAL_CONFIG)

## So sánh với GSD-aug-Tiny

In [ ]:
# GSD-aug-Tiny (train TRÊN Tiny, checkpoint epoch1) trên CÙNG evaluator, pooled 7.000 ảnh/case.
BASE_TINY = {  # case: (BA, AUC, real_recall, fake_recall)
    'native': (0.998571, 0.999986, 0.997429, 0.999714), 'fake_jpeg96': (0.996714, 0.999932, 0.997429, 0.996000),
    'all_jpeg96': (0.996286, 0.999918, 0.996571, 0.996000), 'all_jpeg90': (0.994857, 0.999906, 0.995714, 0.994000),
    'all_jpeg80': (0.994714, 0.999843, 0.996000, 0.993429), 'resize_square256': (0.922000, 0.975754, 0.917143, 0.926857),
    'resize_square560': (0.947571, 0.994563, 0.990286, 0.904857), 'blur_sigma1': (0.996714, 0.999947, 0.997143, 0.996286),
    'resize256_jpeg96': (0.900857, 0.960380, 0.969429, 0.832286)}
out = Path(EVAL_CONFIG['output_root'])
state = json.loads((out / 'evaluation_state.json').read_text())
print('STATUS:', state['status'], '| completed:', state['completed_cases'], '| remaining:', state.get('remaining_cases'))
if (out / 'metrics.csv').is_file():
    m = pd.read_csv(out / 'metrics.csv')
    rows = []
    for _, r in m[m.generator == 'pooled'].iterrows():
        b = BASE_TINY.get(r.case)
        if b:
            rows.append(dict(case=r.case, BA_new=r.balanced_accuracy, BA_gsdaug_tiny=b[0], dBA=r.balanced_accuracy - b[0],
                             AUC_new=r.auc, AUC_base=b[1], real_recall_new=r.real_recall, real_recall_base=b[2],
                             fake_recall_new=r.fake_recall, fake_recall_base=b[3]))
    display(pd.DataFrame(rows))
    print('Theo generator (native):')
    display(m[(m.case == 'native') & (m.generator != 'pooled')]
            [['generator', 'n', 'balanced_accuracy', 'auc', 'real_recall', 'fake_recall']])
if (out / 'partial_metrics.csv').is_file():
    print('PARTIAL (chưa đủ ảnh, không dùng làm kết quả):'); display(pd.read_csv(out / 'partial_metrics.csv'))

archive_path = out.parent / (out.name + '_results.zip')
with zipfile.ZipFile(archive_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(out.rglob('*')):
        if path.is_file() and path.suffix in ('.csv', '.json'):
            archive.write(path, (Path(out.name) / path.relative_to(out)).as_posix())
from IPython.display import FileLink
display(FileLink(str(archive_path)))